# Bài 25 · RNN trở lại (linear attention & Mamba): nén quá khứ vào một ma trận cố định, rồi đo xem nó nhớ được bao nhiêu

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/25-rnn-tro-lai.ipynb)

Notebook đi kèm [Bài 25 của Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rnn-tro-lai/) (reel và demo tương tác nằm ở trang bài học).
Attention giữ khóa và giá trị của mọi token đã đọc, nên bộ nhớ lớn dần theo độ dài văn bản. Bỏ softmax đi, cùng phép
tính đó viết lại được thành một mạng hồi tiếp với **trạng thái cố định** $S$ cỡ $d_k \times d_v$. Câu hỏi trung tâm:
một ma trận như thế nhớ được bao nhiêu, và các lớp hiện đại (cổng quên, quy tắc delta, Mamba-2) quản lý nó thế nào?

**Bạn sẽ làm:**
1. Tính bộ nhớ đệm KV của Llama từ cấu hình đã công bố, và thấy nó phình theo độ dài.
2. Viết linear attention theo ba cách (hồi tiếp, ma trận có mặt nạ, theo khối) và kiểm tra chúng trùng nhau.
3. Ghi $N$ cặp khóa–giá trị vào một ma trận $16 \times 16$ chưa huấn luyện, đo xem đọc lại được bao nhiêu, rồi so với 7 lớp
   đã huấn luyện.
4. Thấy quy tắc delta là một bước hồi quy bình phương tối thiểu trực tuyến, và huấn luyện một lớp có cổng tự học quên.
5. Kiểm tra Mamba-2 chính là linear attention có cổng, rồi tính bộ nhớ của các mô hình lai ở 262.144 token.

**Cần biết trước:** [Bài 15 · RNN & LSTM](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/rnn/) (trạng thái ẩn, cổng
quên), [Bài 16 · Attention](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/attention/) (truy vấn, khóa, giá trị) và
[Bài 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/transformer/) (mặt nạ nhân quả, bộ nhớ đệm
KV). **Thời gian chạy:** dưới 1 phút trên CPU, kể cả tải hai tệp kết quả (~0,1 MB) lần đầu.

Mọi con số trong notebook đều do chính các ô bên dưới tính ra (hoặc đọc từ tệp kết quả mà demo dùng), và trùng với
số trong demo.

> Các ô **🤔 Dự đoán trước** đặt một câu hỏi trước khi chạy. Hãy tự trả lời rồi mới chạy ô kế tiếp.

## 1 · Chuẩn bị

Chỉ cần numpy và matplotlib (Colab đã cài sẵn). Hàm `vn()` in số theo kiểu Việt Nam: dấu phẩy thập phân,
dấu chấm hàng nghìn. Python vẫn *gõ* số thập phân bằng dấu chấm (`0.1`), chỉ phần in ra là đổi. `sci()` in số rất nhỏ
hoặc rất lớn dạng $4{,}6 \cdot 10^{-14}$.

In [ ]:
import hashlib
import json
import math
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter

# Bảng màu của khóa học (an toàn cho người mù màu); luôn đi kèm một mã hóa thứ hai (hình dạng, nét gạch, nhãn)
BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
DIVERGING = LinearSegmentedColormap.from_list("cam_xam_xanh", [ORANGE, "#f2f2f2", BLUE])  # âm → 0 → dương
SEQUENTIAL = LinearSegmentedColormap.from_list("trang_xanh", ["#ffffff", BLUE])  # 0 → 1
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def vn(x, digits=2):
    """Số kiểu Việt Nam: 13.007 · 0,1 · −4,38 (không in “−0”)."""
    s = f"{round(float(x), digits) + 0.0:,.{digits}f}".replace(",", "_").replace(".", ",").replace("_", ".")
    return s.replace("-", "−")


def pct(p, digits=2):
    return vn(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def sci(x, digits=1):
    """Số dạng khoa học kiểu Việt Nam: 4,6·10⁻¹⁴."""
    if x == 0:
        return "0"
    e = math.floor(math.log10(abs(x)))
    m = x / 10.0**e
    if round(abs(m), digits) >= 10:
        m, e = m / 10, e + 1
    return f"{vn(m, digits)}·10{str(e).translate(SUP)}"


def vn_tick(v, _pos=None):
    """Nhãn vạch trên trục: 0,5 · −2 · 60.000 (bỏ số 0 thừa ở cuối)."""
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return vn(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    """Nhãn vạch trục log: 0,5 · 20 · 10⁻¹²."""
    if 0.01 <= v <= 1000:
        return vn_tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{vn_tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def vn_axes(fig):
    """Đổi nhãn mọi trục của hình (cả thanh màu) sang kiểu Việt Nam; gọi ngay trước plt.show().
    Trục log lấy vạch 1–2–5 nếu ngắn, hoặc các lũy thừa của 10 (cách đều, tối đa ~9 vạch) nếu dài."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                # only axes still on matplotlib's default number formatter: text labels and custom formatters stay
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(vn_tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
                if step > 1:
                    axis.set_minor_locator(NullLocator())
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))


print(vn(262144, 0), vn(0.9995, 4), pct(0.7902), sci(4.64e-14))

## 2 · Dữ liệu: hai tệp kết quả của demo

Bài này không cần tập dữ liệu ngoài: các dãy khóa–giá trị được sinh ngẫu nhiên ngay trong notebook. Nhưng hai thứ
chạy quá lâu để làm lại ở đây, nên ta nạp đúng tệp mà demo trên web dùng (có kiểm tra MD5):
- `linear-recall.json`: 74 cấu hình lớp trộn token × 3 seed, mỗi lần 6.000 bước huấn luyện trên bài tra cứu
  khóa–giá trị (khoảng 400 phút CPU), cùng các cổng mà mô hình seed 0 học được và bảng bộ nhớ chưa huấn luyện;
- `linear-cost.json`: số lớp, số đầu, kích thước trạng thái của 10 mô hình thật, đọc từ `config.json` đã công bố.

Ở mỗi phần, notebook chạy trực tiếp một phiên bản nhỏ của cơ chế trước khi đọc kết quả đầy đủ.

In [ ]:
# Kết quả của một lần huấn luyện dài (cùng tệp với demo trên web). build.py thay COURSE_DATA_URL bằng thư mục data/
# của repo notebook công khai.
def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} bị hỏng, hãy xóa và tải lại"
    return json.loads(p.read_text())


REC = fetch_json("linear-recall.json", "978ff84d1924cb1d79e398eaaea8e026")  # kết quả huấn luyện của demo
COST = fetch_json("linear-cost.json", "fa39c9c60a611e5ebf6f432ae4878a7f")  # cấu hình mô hình của demo
print(
    f"linear-recall.json: {len(REC['runs'])} cấu hình × {len(REC['setup']['seeds'])} seed, "
    f"{vn(REC['setup']['steps'], 0)} bước mỗi lần; {vn(REC['setup']['nVal'], 0)} dãy kiểm định, "
    f"{vn(REC['setup']['nTest'], 0)} dãy kiểm tra"
)
print("linear-cost.json:", ", ".join(m["name"] for m in COST["models"]))
size = sum((Path("data") / n).stat().st_size for n in ("linear-recall.json", "linear-cost.json"))
print(f"tổng cộng {vn(size / 1e6, 1)} MB")

## 3 · Bộ nhớ đệm KV phình theo độ dài

Khi sinh văn bản, mỗi lớp attention giữ khóa $k$ và giá trị $v$ của mọi token đã đọc (Bài 17), để token mới khỏi
phải tính lại chúng. Mỗi token, mỗi lớp thêm $2 \cdot n_{kv} \cdot d_{head}$ số vào bộ nhớ đệm: K và V của từng đầu
KV. Ta lấy cấu hình đã công bố của hai mô hình 32 lớp, mỗi số lưu ở bf16 (2 byte):
- **Llama 2 7B** (Touvron và cộng sự, 2023): 32 đầu truy vấn, 32 đầu KV, mỗi đầu 128 chiều;
- **Llama 3 8B** (Grattafiori và cộng sự, 2024): 32 đầu truy vấn nhưng chỉ 8 đầu KV (grouped-query attention, GQA:
  4 đầu truy vấn dùng chung một cặp K, V).

### 🤔 Dự đoán trước
Llama 3 8B lớn hơn Llama 2 7B một chút. Ở 262.144 token, bộ nhớ đệm KV của mô hình nào lớn hơn, và gấp mấy lần?

In [ ]:
KiB, MiB, GiB = 2**10, 2**20, 2**30
T_LONG = 262_144  # 2¹⁸ token ("256K"), ngữ cảnh gốc của Jamba và Qwen3-Next

LLAMA = {  # config.json đã công bố; Llama 2 chỉ dùng GQA ở bản 34B và 70B
    "Llama 3 8B": {"layers": 32, "heads": 32, "kv_heads": 8, "head_dim": 128},
    "Llama 2 7B": {"layers": 32, "heads": 32, "kv_heads": 32, "head_dim": 128},
}


def kv_per_token(attn_layers, kv_heads, head_dim, bytes_per_number=2):
    """Số byte mỗi token thêm vào bộ nhớ đệm: K và V của mọi đầu KV, ở mọi lớp attention (bf16 = 2 byte mỗi số)."""
    return attn_layers * 2 * kv_heads * head_dim * bytes_per_number


for name, c in LLAMA.items():
    per = kv_per_token(c["layers"], c["kv_heads"], c["head_dim"])
    sizes = " · ".join(f"{vn(T, 0)} token: {vn(per * T / GiB, 1)} GiB" for T in (4096, 32768, T_LONG, 4 * T_LONG))
    print(f"{name}: {vn(per / KiB, 0)} KiB mỗi token  →  {sizes}")
kv3, kv2 = (kv_per_token(c["layers"], c["kv_heads"], c["head_dim"]) for c in LLAMA.values())
assert kv3 == 128 * KiB and kv3 * T_LONG == 32 * GiB, "khác với bài học: hãy kiểm tra lại cấu hình"
assert kv2 * T_LONG == 128 * GiB  # đúng con số 128GB của LLaMA-2 trong Bảng 1 của bài báo Jamba
reads = T_LONG * (T_LONG + 1) // 2
print(f"sinh {vn(T_LONG, 0)} token: token thứ t đọc lại t token đã lưu → tổng {sci(reads)} lần đọc (tăng theo T²)")

Llama 2 7B cần gấp 4 lần: 128 GiB, so với 32 GiB của Llama 3 8B, dù ít tham số hơn. Lý do là GQA: Llama 3 chỉ giữ 8 cặp
K, V mỗi lớp thay vì 32. Con số 128 GiB của Llama 2 trùng với Bảng 1 của bài báo Jamba (Lieber và cộng sự, 2024), ghi
là “128GB” ở 256K token và 16 bit.

Và bộ nhớ chỉ là một nửa vấn đề. Mỗi token mới phải đọc lại *cả* bộ nhớ đệm: token thứ $t$ đọc $t$ cặp $(k, v)$, nên
sinh $T$ token tốn tổng cộng khoảng $T^2/2$ lần đọc. GQA chỉ chia nhỏ hằng số; bộ nhớ vẫn tăng tuyến tính theo $T$.

## 4 · Bỏ softmax: cùng một phép tính, viết theo hai cách

Attention có mặt nạ nhân quả (Bài 17), theo quy ước hàng ($q_t, k_t$ là vectơ hàng $1 \times d_k$, $v_t$ là
$1 \times d_v$):

$$o_t = \frac{\sum_{s \le t} \exp(q_t k_s^\top/\sqrt{d_k})\, v_s}{\sum_{s \le t} \exp(q_t k_s^\top/\sqrt{d_k})}$$

Katharopoulos và cộng sự (2020) thay $\exp(q k^\top)$ bằng $\varphi(q)\varphi(k)^\top$ với $\varphi(x) = \operatorname{elu}(x) + 1$
(luôn dương). Khi đó $\varphi(q_t)$ ra khỏi tổng được, nhờ tính kết hợp của phép nhân ma trận:

$$o_t = \frac{\varphi(q_t) \sum_{s \le t} \varphi(k_s)^\top v_s}{\varphi(q_t) \sum_{s \le t} \varphi(k_s)^\top}
= \frac{\varphi(q_t)\, S_t}{\varphi(q_t)\, z_t^\top}, \qquad S_t = S_{t-1} + \varphi(k_t)^\top v_t, \quad z_t = z_{t-1} + \varphi(k_t)$$

Viết theo kiểu attention, ta tính ma trận $T \times T$ có mặt nạ $\big(\varphi(Q)\varphi(K)^\top \odot M\big)V$ rồi chia cho
tổng mỗi hàng. Viết theo kiểu RNN, ta chỉ mang theo một **trạng thái** $S_t$ cỡ $d_k \times d_v$ (và $z_t$ cỡ $1 \times d_k$),
cập nhật mỗi token một lần. Ô dưới tính cả hai cách, và cả softmax attention để so sánh, trên một dãy ngẫu nhiên.

In [ ]:
def elu1(x):
    """φ(x) = elu(x) + 1: luôn dương, nên tổng ở mẫu số không bao giờ bằng 0."""
    return np.where(x > 0, x + 1, np.exp(np.minimum(x, 0)))


def softmax_attention(Q, K, V):
    s = Q @ K.T / np.sqrt(Q.shape[1])
    s = np.where(np.tri(len(Q), dtype=bool), s, -np.inf)  # mặt nạ nhân quả: token t chỉ nhìn s ≤ t
    P = np.exp(s - s.max(1, keepdims=True))
    return (P / P.sum(1, keepdims=True)) @ V


def linear_attention_matrix(Q, K, V):
    A = (elu1(Q) @ elu1(K).T) * np.tri(len(Q))  # ma trận T × T có mặt nạ, không softmax
    return (A @ V) / A.sum(1, keepdims=True)


def linear_attention_rnn(Q, K, V):
    S, z = np.zeros((K.shape[1], V.shape[1])), np.zeros(K.shape[1])  # trạng thái cố định: d_k × d_v và 1 × d_k
    O = np.zeros_like(V)
    for t in range(len(Q)):
        S += np.outer(elu1(K[t]), V[t])  # S_t = S_{t−1} + φ(k_t)ᵀ v_t
        z += elu1(K[t])
        O[t] = elu1(Q[t]) @ S / (elu1(Q[t]) @ z)
    return O


rng = np.random.default_rng(0)
T, d = 64, 16
Q, K, V = rng.standard_normal((3, T, d))
O_mat, O_rnn, O_soft = linear_attention_matrix(Q, K, V), linear_attention_rnn(Q, K, V), softmax_attention(Q, K, V)
print(f"dạng ma trận và dạng RNN lệch nhau nhiều nhất {sci(np.abs(O_mat - O_rnn).max())}")
print(f"linear attention và softmax attention lệch nhau trung bình {vn(np.abs(O_mat - O_soft).mean(), 3)}")
print(f"bộ nhớ ở token cuối: softmax giữ {vn(T * 2 * d, 0)} số (T·2d), dạng RNN giữ {vn(d * d + d, 0)} số (d² + d)")
assert np.abs(O_mat - O_rnn).max() < 1e-10

Hai cách trùng nhau đến sai số làm tròn, nhưng khác hẳn softmax attention: đây là một mô hình *khác*, không phải một
cách tính nhanh hơn của cùng mô hình. Từ token thứ $d/2 + 1 = 9$, bộ nhớ đệm $T \cdot 2d$ của softmax đã lớn hơn trạng thái
$d \times d$; ở 64 token nó lớn gấp gần 8 lần, và còn lớn mãi.

### Dạng tổng quát: cổng quên $\alpha_t$ và độ mạnh ghi $\beta_t$

Các lớp hiện nay bỏ $\varphi$ và mẫu số (họ delta còn chuẩn hóa $q, k$ về độ dài 1; ở đây ta làm vậy cho mọi lớp), và thêm
hai số cho mỗi token:

$$S_t = \alpha_t\, S_{t-1} + \beta_t\, k_t^\top v_t, \qquad o_t = q_t S_t$$

- $\alpha_t \in (0, 1)$ là **cổng quên**: co cả trạng thái lại trước khi ghi (giống $f_t$ của LSTM ở Bài 15, nhưng chỉ là
  một số cho mỗi đầu);
- $\beta_t$ là **độ mạnh khi ghi**.

Thế ngược $S_{t-1}$ vào ta được dạng attention: $o_t = \sum_{s \le t} \big(\prod_{r=s+1}^{t} \alpha_r\big)\, \beta_s\,
(q_t k_s^\top)\, v_s$. Viết thành ma trận: $O = (QK^\top \odot M)\operatorname{diag}(\beta)V$, trong đó mặt nạ $M_{ts} =
\prod_{r=s+1}^{t} \alpha_r$ với $s \le t$ (và 0 nếu $s > t$). Với $\alpha = 1$, $M$ chính là mặt nạ nhân quả của Bài 17.

Còn một cách thứ ba, cách mà GPU dùng khi huấn luyện: **chia thành khối** $C$ token. Trong mỗi khối tính như attention có
mặt nạ (nhân ma trận), giữa các khối chỉ chuyền trạng thái $S$. RNN của Bài 15 không làm vậy được vì $\tanh$ nằm trong
vòng lặp; ở đây $S_t$ phụ thuộc tuyến tính vào $S_{t-1}$, nên tách được.

Ô dưới sinh dãy giống hệt demo (bộ sinh số ngẫu nhiên `mulberry32` của JavaScript, chuyển sang numpy, và phép Box–Muller),
để kết quả trùng với những gì trình duyệt tính. Hàm `sequence` có bốn kiểu cổng: không cổng, $\alpha$ cố định, $\alpha$ theo
token (thỉnh thoảng gần 0, như một lần “xóa”), và kiểu dài cho phép thử 2.048 token.

In [ ]:
def mulberry32(seed, n):
    """n số đầu tiên của bộ sinh mulberry32(seed) trong JavaScript (demo dùng đúng bộ sinh này)."""
    a = ((seed + np.arange(1, n + 1, dtype=np.uint64) * 0x6D2B79F5) & 0xFFFFFFFF).astype(np.uint32)
    t = (a ^ (a >> 15)) * (a | 1)
    t = (t + (t ^ (t >> 7)) * (t | 61)) ^ t
    return (t ^ (t >> 14)).astype(np.float64) / 2**32


def box_muller(u):
    """Mỗi cặp số đều (u₁, u₂) cho một số chuẩn N(0, 1)."""
    return np.sqrt(-2 * np.log(1 - u[..., 0::2])) * np.cos(2 * np.pi * u[..., 1::2])


def unit(x):
    return x / np.linalg.norm(x, axis=-1, keepdims=True)


def sequence(seed, T, dk, dv, gate, alpha=0.9):
    """Mỗi token, theo thứ tự: q (d_k số chuẩn), k (d_k), v (d_v), rồi hai số đều g₁, g₂ cho cổng."""
    u = mulberry32(seed, T * (4 * dk + 2 * dv + 2)).reshape(T, -1)
    Q, K = unit(box_muller(u[:, : 2 * dk])), unit(box_muller(u[:, 2 * dk : 4 * dk]))
    V = box_muller(u[:, 4 * dk : 4 * dk + 2 * dv])
    g1, g2 = u[:, -2], u[:, -1]
    a = {
        "none": np.ones(T),
        "fixed": np.full(T, alpha),
        "token": np.where(g1 < 0.08, 0.05 + 0.1 * g2, 0.9 + 0.1 * g2),  # 8% token “xóa” (α < 0,15)
        "long": 0.9 + 0.1 * g1,
    }[gate]
    b = 0.5 + 0.5 * g2 if gate == "long" else np.ones(T)
    return Q, K, V, a, b


Q, K, V, a, b = sequence(7, 64, 4, 4, "token")
print(
    "q₁ =", "; ".join(vn(x, 3) for x in Q[0]), " (độ dài 1)  ·  α của 12 token đầu:", " ".join(vn(x, 2) for x in a[:12])
)

Ba hàm dưới tính cùng một $O$ theo ba cách. Hàm `decay` dựng mặt nạ $M$ từ tổng tích lũy $c_t = \sum_{u \le t} \log\alpha_u$,
vì $\prod_{r=s+1}^{t} \alpha_r = e^{c_t - c_s}$. Ở dạng theo khối, token $t$ của một khối đọc trạng thái mang từ khối
trước (đã co thêm $e^{c_t}$) cộng với phần attention có mặt nạ bên trong khối; cuối khối, trạng thái được cập nhật một lần
bằng một phép nhân ma trận.

In [ ]:
def recurrent(Q, K, V, a, b):
    """Dạng RNN: một trạng thái d_k × d_v, cập nhật từng token."""
    S, O = np.zeros((K.shape[1], V.shape[1])), np.zeros_like(V)
    for t in range(len(a)):
        S = a[t] * S + b[t] * np.outer(K[t], V[t])  # S_t = α_t S_{t−1} + β_t k_tᵀ v_t
        O[t] = Q[t] @ S  # o_t = q_t S_t
    return O


def decay(a):
    """c_t = Σ_{u≤t} log α_u và mặt nạ M_ts = ∏_{r=s+1..t} α_r = exp(c_t − c_s) khi s ≤ t, 0 khi s > t."""
    c = np.cumsum(np.log(a))
    return c, np.exp(np.where(np.tri(len(a), dtype=bool), c[:, None] - c[None, :], -np.inf))


def parallel(Q, K, V, a, b):
    """Dạng attention: ma trận T × T có mặt nạ, O = (QKᵀ ⊙ M) diag(β) V."""
    _, M = decay(a)
    return ((Q @ K.T) * M * b[None, :]) @ V


def chunked(Q, K, V, a, b, C):
    """Dạng theo khối C token: có mặt nạ bên trong khối, chuyền trạng thái giữa các khối."""
    S, O = np.zeros((K.shape[1], V.shape[1])), np.zeros_like(V)
    for t0 in range(0, len(a), C):
        blk = slice(t0, min(len(a), t0 + C))
        c, M = decay(a[blk])
        O[blk] = np.exp(c)[:, None] * (Q[blk] @ S) + ((Q[blk] @ K[blk].T) * M * b[None, blk]) @ V[blk]
        S = np.exp(c[-1]) * S + (K[blk] * (np.exp(c[-1] - c) * b[blk])[:, None]).T @ V[blk]
    return O


# (seed, T, d_k, d_v, cổng, α, khối C): các dãy của demo (d 4, T ≤ 64) và phép thử 2.048 token (d 16, C 64)
CASES = [
    (7, 64, 4, 4, "none", 0.9, 8),
    (7, 64, 4, 4, "fixed", 0.8, 8),
    (7, 64, 4, 4, "token", 0.9, 16),
    (3, 37, 4, 4, "token", 0.9, 4),
    (1, 2048, 16, 16, "long", 0.9, 64),
]
for seed, T, dk, dv, gate, alpha, C in CASES:
    s = sequence(seed, T, dk, dv, gate, alpha)
    O = recurrent(*s)
    e_par, e_chk = np.abs(O - parallel(*s)).max(), np.abs(O - chunked(*s, C)).max()
    print(
        f"T = {T:>5} · d = {dk:>2} · cổng {gate:<5}: |o| tới {vn(np.abs(O).max(), 2):>5}; "
        f"ma trận lệch {sci(e_par)}, theo khối (C = {C}) lệch {sci(e_chk)}"
    )
    assert max(e_par, e_chk) < 1e-10, "ba dạng phải trùng nhau: hãy kiểm tra lại code"

Ba cách cho cùng kết quả ở mọi kiểu cổng. Chỗ lệch nhiều nhất ở 2.048 token chỉ cỡ $10^{-14}$ (dạng ma trận cộng 2.048
số hạng một lúc, dạng hồi tiếp cộng dồn theo thứ tự khác), còn dạng theo khối lệch cỡ $10^{-15}$. Đó là sai số làm tròn
của số thực 64 bit, không phải sai số của phương pháp.

Hình dưới vẽ dãy có cổng theo token ($T = 64$). Bên trái là mặt nạ suy giảm $M_{ts}$: mỗi hàng $t$ cho biết token $s$ cũ
còn lại bao nhiêu khi token $t$ đọc trạng thái. Những token “xóa” ($\alpha_t < 0{,}15$, vạch xám trên trục) cắt mặt nạ thành
các khối gần như độc lập. Bên phải là trọng số thật $(q_t \cdot k_s)\,M_{ts}\,\beta_s$ của dạng ma trận: giống bản đồ attention
của Bài 16, nhưng không có softmax nên có cả trọng số âm (cam) lẫn dương (xanh), và các hàng không cộng lại bằng 1.

In [ ]:
Q, K, V, a, b = sequence(7, 64, 4, 4, "token")
_, M = decay(a)
W = (Q @ K.T) * M * b[None, :]
resets = np.flatnonzero(a < 0.15)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6))
im0 = axes[0].imshow(M, cmap=SEQUENTIAL, vmin=0, vmax=1)
vmax = np.abs(W).max()
im1 = axes[1].imshow(W, cmap=DIVERGING, vmin=-vmax, vmax=vmax)
for ax, im, title in ((axes[0], im0, "mặt nạ suy giảm M (tích các cổng α)"), (axes[1], im1, "trọng số (qₜ·kₛ)·Mₜₛ·βₛ")):
    ax.set(xlabel="token được đọc s", ylabel="token đang đọc t", title=title)
    ax.plot(resets, np.full(len(resets), 63.5), "|", color=GREY, ms=14, mew=2, clip_on=False)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
vn_axes(fig)
plt.show()
print(
    f"{len(resets)} token “xóa” ở vị trí {', '.join(map(str, resets))}; α của chúng: {', '.join(vn(x, 3) for x in a[resets])}"
)

### Quy tắc delta trong ba dạng

Linear attention chỉ *cộng thêm* vào trạng thái. Quy tắc delta (Schlag, Irie & Schmidhuber, 2021) ghi phần **sai khác**
giữa giá trị mới và giá trị mà trạng thái đang giữ cho khóa đó:

$$u_t = v_t - \alpha_t\, k_t S_{t-1}, \qquad S_t = \alpha_t S_{t-1} + \beta_t\, k_t^\top u_t
\quad\Longleftrightarrow\quad S_t = \alpha_t \big(I - \beta_t k_t^\top k_t\big) S_{t-1} + \beta_t\, k_t^\top v_t$$

Với $\alpha = 1$ đó là DeltaNet, với $\alpha_t$ theo token là Gated DeltaNet (Yang, Kautz & Hatamizadeh, 2024). Phép cập
nhật vẫn tuyến tính theo $S$, nên vẫn có dạng ma trận, chỉ cần thay $V$ bằng $U$: thế $S_{t-1}$ vào $u_t$ cho
$u_t + \sum_{s<t} M_{ts}\,\beta_s\,(k_t \cdot k_s)\, u_s = v_t$, tức một hệ tam giác dưới $(I + L)U = V$ với
$L_{ts} = M_{ts}\beta_s (k_t \cdot k_s)$ khi $s < t$ (dạng “UT” mà DeltaNet dùng để huấn luyện theo khối).

In [ ]:
def recurrent_delta(Q, K, V, a, b):
    S, O = np.zeros((K.shape[1], V.shape[1])), np.zeros_like(V)
    for t in range(len(a)):
        u = V[t] - a[t] * (K[t] @ S)  # phần sai khác: v_t trừ giá trị đang lưu cho k_t
        S = a[t] * S + b[t] * np.outer(K[t], u)
        O[t] = Q[t] @ S
    return O


def solve_u(K, V, b, M, R=0):
    """(I + L) U = V − R, với L_ts = M_ts β_s (k_t·k_s) khi s < t (tam giác dưới, đường chéo bằng 1)."""
    L = np.tril(M * b[None, :] * (K @ K.T), -1)
    return np.linalg.solve(np.eye(len(b)) + L, V - R)


def parallel_delta(Q, K, V, a, b):
    _, M = decay(a)
    return ((Q @ K.T) * M * b[None, :]) @ solve_u(K, V, b, M)


def chunked_delta(Q, K, V, a, b, C):
    S, O = np.zeros((K.shape[1], V.shape[1])), np.zeros_like(V)
    for t0 in range(0, len(a), C):
        blk = slice(t0, min(len(a), t0 + C))
        c, M = decay(a[blk])
        e = np.exp(c)
        U = solve_u(K[blk], V[blk], b[blk], M, e[:, None] * (K[blk] @ S))  # trừ phần trạng thái mang từ khối trước
        O[blk] = e[:, None] * (Q[blk] @ S) + ((Q[blk] @ K[blk].T) * M * b[None, blk]) @ U
        S = np.exp(c[-1]) * S + (K[blk] * (np.exp(c[-1] - c) * b[blk])[:, None]).T @ U
    return O


for seed, T, dk, dv, gate, alpha, C in CASES:
    s = sequence(seed, T, dk, dv, gate, alpha)
    O = recurrent_delta(*s)
    e_par, e_chk = np.abs(O - parallel_delta(*s)).max(), np.abs(O - chunked_delta(*s, C)).max()
    print(f"delta, T = {T:>5} · cổng {gate:<5}: ma trận lệch {sci(e_par)}, theo khối (C = {C}) lệch {sci(e_chk)}")
    assert max(e_par, e_chk) < 1e-10, "ba dạng phải trùng nhau: hãy kiểm tra lại code"

Quy tắc delta cũng cho ba dạng trùng nhau đến sai số làm tròn. Cái giá của nó là một phép giải hệ tam giác $C \times C$
trong mỗi khối, thay vì chỉ nhân ma trận.

## 5 · Một ma trận 16 × 16 nhớ được bao nhiêu cặp khóa–giá trị?

Trạng thái không lớn thêm, nên chắc chắn có lúc nó đầy. Để đo điều đó, ta dùng một bộ nhớ **chưa huấn luyện**, đúng như
thẻ thứ hai của demo:
- một bảng 64 “từ” giá trị, mỗi từ là một vectơ đơn vị ngẫu nhiên trong $\mathbb{R}^{16}$;
- $N$ cặp: mỗi cặp có một khóa đơn vị ngẫu nhiên $k_i$ ($1 \times 16$) và một từ giá trị $v_i$ chọn ngẫu nhiên;
- ghi cả $N$ cặp vào $S$ ($16 \times 16$), rồi hỏi lại từng khóa: $o_i = k_i S$, và đoán là từ có tích vô hướng với $o_i$
  lớn nhất. Đúng nếu ra đúng $v_i$.

Ta lấy trung bình trên 400 lần rút ngẫu nhiên, với cùng các số ngẫu nhiên mà demo dùng.

Bốn cách ghi:
- **cộng:** $S \leftarrow S + k_i^\top v_i$ (linear attention);
- **có cổng:** $S \leftarrow 0{,}9\, S + k_i^\top v_i$ (cổng quên cố định $\alpha = 0{,}9$);
- **delta:** $S \leftarrow S + k_i^\top (v_i - k_i S)$ ($\beta = 1$);
- **bình phương tối thiểu:** ma trận $S$ làm $\sum_i \lVert k_i S - v_i \rVert^2$ nhỏ nhất, tính từ mọi cặp cùng lúc: ma trận
  khớp các giá trị tốt nhất theo sai số bình phương.

In [ ]:
CODEWORDS = 64  # số từ giá trị


def memory_draws(draws=400, dk=16, nmax=64):
    """Như demo: lần rút d dùng mulberry32(1 + d); trước là 64 từ giá trị, sau là từng cặp (khóa d_k số, rồi chỉ số từ)."""
    per = 2 * dk + 1
    U = np.stack([mulberry32(1 + d, 2 * CODEWORDS * dk + nmax * per) for d in range(draws)])
    code = unit(box_muller(U[:, : 2 * CODEWORDS * dk]).reshape(draws, CODEWORDS, dk))
    pairs = U[:, 2 * CODEWORDS * dk :].reshape(draws, nmax, per)
    keys = unit(box_muller(pairs[..., : 2 * dk]))  # (lần rút, cặp, d_k), độ dài 1
    ids = np.floor(pairs[..., 2 * dk] * CODEWORDS).astype(int)
    vals = np.take_along_axis(code, ids[..., None], 1)  # v_i là từ số ids[i]
    return code, keys, ids, vals


def read_back(S, code, K, ids):
    """Hỏi lại mọi khóa: o = kS, đoán là từ gần o nhất (tích vô hướng lớn nhất). Trả về mảng đúng/sai."""
    O = K @ S
    return (O @ code.transpose(0, 2, 1)).argmax(-1) == ids


def write(rule, K, V, alpha=0.9):
    """Ghi các cặp (hàng của K, V) vào S, cho mọi lần rút cùng lúc."""
    S = np.zeros((len(K), K.shape[2], V.shape[2]))
    if rule == "ls":
        Kt = K.transpose(0, 2, 1)
        if K.shape[1] <= K.shape[2]:  # ít cặp hơn số chiều: khớp đúng mọi cặp, S = Kᵀ(KKᵀ)⁻¹V
            return Kt @ np.linalg.solve(K @ Kt, V)
        return np.linalg.solve(Kt @ K, Kt @ V)  # nhiều cặp hơn: S = (KᵀK)⁻¹KᵀV
    for i in range(K.shape[1]):
        k, v = K[:, i], V[:, i]
        if rule == "delta":
            v = v - np.einsum("bi,bij->bj", k, S)  # chỉ ghi phần sai khác
        S = (alpha if rule == "gate" else 1.0) * S + k[:, :, None] * v[:, None, :]
    return S


code, keys, ids, vals = memory_draws()
N = 4
S = write("add", keys[:1, :N], vals[:1, :N])
ok = read_back(S, code[:1], keys[:1, :N], ids[:1, :N])[0]
print(
    f"lần rút 0, {N} cặp: S cỡ {S.shape[1]} × {S.shape[2]}; giá trị cần đọc {ids[0, :N].tolist()}, đọc đúng {ok.sum()}/{N}"
)
print(
    f"tích vô hướng giữa các khóa: {' '.join(vn(x, 2) for x in (keys[0, :N] @ keys[0, :N].T)[np.triu_indices(N, 1)])}"
)

Đọc lại một khóa cho $o_i = k_i S = \sum_j (k_i \cdot k_j)\, v_j = v_i + \sum_{j \ne i} (k_i \cdot k_j)\, v_j$: giá trị đúng
cộng với **nhiễu chồng lấn** từ mọi cặp khác. Khóa ngẫu nhiên trong 16 chiều không vuông góc nhau (tích vô hướng cỡ
$\pm 0{,}25$), nên càng nhiều cặp, nhiễu càng lớn.

### 🤔 Dự đoán trước
Ma trận $16 \times 16$ có đúng 16 hướng độc lập. Ghi **16 cặp** với khóa ngẫu nhiên bằng phép cộng: đọc lại đúng khoảng bao
nhiêu phần trăm? Và quy tắc delta, vốn “sửa” giá trị trước khi ghi, đúng nhiều hơn hay ít hơn phép cộng?

In [ ]:
RULES = {"add": "cộng kᵀv", "gate": "có cổng, α = 0,9", "delta": "quy tắc delta", "ls": "bình phương tối thiểu"}
NS = np.arange(1, 65)


def capacity(rule, keys, vals, code, ids):
    """Tỉ lệ đọc đúng khi ghi N = 1…64 cặp (trung bình trên mọi lần rút) và mảng đúng/sai của từng N."""
    acc, hits = [], []
    for N in NS:
        S = write(rule, keys[:, :N], vals[:, :N])
        h = read_back(S, code, keys[:, :N], ids[:, :N])
        acc.append(h.mean())
        hits.append(h)
    return np.array(acc), hits


table, hits = {}, {}
for rule in RULES:
    table[rule], hits[rule] = capacity(rule, keys, vals, code, ids)
for N in (8, 16, 32, 64):
    print(f"N = {N:>2}: " + " · ".join(f"{RULES[r]} {pct(table[r][N - 1], 1)}" for r in RULES))
stored = REC["untrained"]["rand"]  # bảng mà demo tính bằng JavaScript với cùng các lần rút
worst = max(np.abs(table[r] - np.array(stored[r]["acc"])).max() for r in RULES)
print(f"so với bảng của demo (làm tròn 4 chữ số): lệch nhiều nhất {vn(100 * worst, 3)} điểm phần trăm")
assert worst < 1e-3, "khác với demo: hãy kiểm tra lại các lần rút ngẫu nhiên"

Hình dưới vẽ tỉ lệ đọc đúng theo số cặp $N$ cho bốn cách ghi; mỗi cách có màu và kiểu điểm riêng. Đường gạch đứng đánh
dấu $N = d_k = 16$.

In [ ]:
STYLE = {"add": (BLUE, "o", "-"), "gate": (ORANGE, "^", "--"), "delta": (GREEN, "D", "-"), "ls": ("black", "s", ":")}
fig, ax = plt.subplots(figsize=(8, 3.8))
for rule, label in RULES.items():
    color, marker, ls = STYLE[rule]
    ax.plot(NS, 100 * table[rule], color=color, marker=marker, ls=ls, ms=4, markevery=4, lw=1.8, label=label)
ax.axvline(16, color=GREY, lw=1, ls="--")
ax.text(16.8, 6, "N = d_k = 16", color=GREY, fontsize=9)
ax.set(xlabel="số cặp N đã ghi", ylabel="đọc lại đúng (%)", xlim=(1, 64), ylim=(0, 102))
ax.set_title("Bộ nhớ 16 × 16 chưa huấn luyện, khóa ngẫu nhiên (trung bình 400 lần rút)")
ax.legend(frameon=False, loc="upper right")
ax.grid(alpha=0.3)
vn_axes(fig)
plt.show()

Phép cộng chỉ đúng 83,1% ở 16 cặp, dù ma trận có đủ 16 hướng: khóa ngẫu nhiên chồng lên nhau, và nhiễu chồng lấn cộng dồn.
Nếu các khóa **trực giao**, $k_i \cdot k_j = 0$ với $i \ne j$, thì $o_i = v_i$ đúng tuyệt đối, đến tối đa $d_k$ cặp. Ô dưới
trực giao hóa các khóa (Gram–Schmidt, bằng `np.linalg.qr`) rồi ghi lại.

In [ ]:
N = 16
Qr = np.linalg.qr(keys[:, :N].transpose(0, 2, 1))[0]  # các cột trực chuẩn, cùng không gian con với N khóa đầu
K_orth = Qr.transpose(0, 2, 1)
for rule in ("add", "delta"):
    acc_orth = read_back(write(rule, K_orth, vals[:, :N]), code, K_orth, ids[:, :N]).mean()
    print(
        f"{N} khóa trực giao, {RULES[rule]}: đúng {pct(acc_orth, 1)}  (khóa ngẫu nhiên: {pct(table[rule][N - 1], 1)})"
    )
    assert acc_orth == 1.0
q_gate = [h.mean() for h in np.array_split(hits["gate"][31], 4, axis=1)]
q_add = [h.mean() for h in np.array_split(hits["add"][31], 4, axis=1)]
q_delta = [h.mean() for h in np.array_split(hits["delta"][31], 4, axis=1)]
print("32 cặp, chia 4 nhóm theo thứ tự ghi (cũ → mới):")
for name, q in (("cộng", q_add), ("có cổng α = 0,9", q_gate), ("delta", q_delta)):
    print(f"  {name:<16} " + " / ".join(pct(x, 0) for x in q))
print(f"bình phương tối thiểu: {pct(table['ls'][31], 1)} ở 32 cặp, {pct(table['ls'][63], 1)} ở 64 cặp")

Ba kiểu thất bại khác nhau:
- **Phép cộng** sai đều ở mọi nhóm (khoảng 53–55% mỗi nhóm khi có 32 cặp): đó là **chồng lấn**, không phải quên.
- **Cổng cố định** $\alpha = 0{,}9$ làm cặp ghi cách đây $n$ bước chỉ còn $0{,}9^n$ độ lớn. Nhóm mới nhất đúng 89%, nhóm cũ
  nhất 3%: đây mới là **quên** thật, và nó không chọn lọc.
- **Quy tắc delta** chưa huấn luyện còn *kém* phép cộng. Mỗi lần ghi, nó sửa $S$ sao cho khóa mới đọc ra đúng giá trị mới;
  phần $S$ bị sửa là hướng của $k_t$, mà hướng đó chồng lên các khóa cũ, nên giá trị cũ bị ghi đè dần. Lỗi của nó cũng
  nghiêng về các cặp cũ.

Nghiệm bình phương tối thiểu khớp đúng mọi cặp khi $N \le 16$, nhưng ở 32 cặp chỉ còn 79,0%. Hệ $KS = V$ có $N$ hàng mà mỗi
cột của $S$ chỉ có $d_k = 16$ ẩn, nên với $N > d_k$ khóa tùy ý không ma trận nào trả lại *đúng y nguyên* mọi giá trị. Trí nhớ
tăng theo $d_k$, không theo độ dài văn bản. (Đọc đúng chỉ cần giá trị đọc ra gần từ đúng hơn mọi từ khác, một điều kiện
lỏng hơn; mục 7 sẽ quay lại chỗ này.)

## 6 · Quy tắc delta là hồi quy trực tuyến

Vì sao gọi là “delta”? Coi $S$ là một mô hình tuyến tính đoán giá trị từ khóa, $\hat v = kS$, và đo sai số trên đúng một cặp:

$$\mathcal{L}_t(S) = \tfrac12 \lVert k_t S - v_t \rVert^2, \qquad \nabla_S \mathcal{L}_t = k_t^\top (k_t S - v_t)$$

Một bước gradient descent (Bài 05) với tốc độ học $\eta = \beta_t$ cho $S - \beta_t k_t^\top (k_t S - v_t) =
S + \beta_t k_t^\top (v_t - k_t S)$: đúng quy tắc delta. Linear attention thì là một bước trên mất mát *tuyến tính*
$-\langle k_t S, v_t\rangle$, có gradient $-k_t^\top v_t$ không phụ thuộc $S$: nó cộng thêm mà không bao giờ sửa
(Yang, 2024, blog *DeltaNet Explained*). Ô dưới kiểm tra gradient bằng sai phân hữu hạn, và kiểm tra rằng với $\beta = 1$
và $\lVert k \rVert = 1$, khóa vừa ghi đọc ra đúng giá trị vừa ghi.

In [ ]:
rng = np.random.default_rng(1)
S0 = rng.standard_normal((16, 16))
k, v = unit(rng.standard_normal(16)), rng.standard_normal(16)
loss = lambda S: 0.5 * np.sum((k @ S - v) ** 2)
grad = np.outer(k, k @ S0 - v)  # ∇_S L = kᵀ(kS − v)
h, i, j = 1e-6, 3, 5
E = np.zeros_like(S0)
E[i, j] = h
fd = (loss(S0 + E) - loss(S0 - E)) / (2 * h)
print(f"∂L/∂S[{i},{j}]: công thức {vn(grad[i, j], 6)} · sai phân hữu hạn {vn(fd, 6)}")
for beta in (1.0, 0.5):
    S1 = S0 - beta * grad
    assert np.allclose(S1, S0 + beta * np.outer(k, v - k @ S0))  # một bước gradient = quy tắc delta
    print(f"β = {vn(beta, 1)}: sai số trên cặp vừa ghi {vn(loss(S0), 3)} → {vn(loss(S1), 3)}")
assert abs(fd - grad[i, j]) < 1e-6 and loss(S0 - grad) < 1e-20

Với $\beta = 1$, sai số trên cặp vừa ghi về đúng 0: quy tắc delta *xóa* giá trị cũ của $k_t$ rồi *ghi* giá trị mới. Với
$\beta = 0{,}5$, sai số còn đúng $(1 - \beta)^2 = \tfrac14$ lần. Nhưng mỗi bước chỉ nhìn một cặp. Nếu lặp lại nhiều lượt trên
cùng $N$ cặp với $\beta$ nhỏ, đó là SGD trên bài hồi quy $\min_S \sum_i \lVert k_i S - v_i \rVert^2$, và nó tiến dần về nghiệm
bình phương tối thiểu của mục 5. Ô dưới thử với 32 cặp ngẫu nhiên, trên cả 400 lần rút.

In [ ]:
N, PASSES = 32, 30
K32, V32 = keys[:, :N], vals[:, :N]
S_ls = write("ls", K32, V32)
mse = lambda S: 0.5 * ((K32 @ S - V32) ** 2).sum(-1).mean()
acc32 = lambda S: read_back(S, code, K32, ids[:, :N]).mean()
curves = {}
for beta in (1.0, 0.3, 0.1):
    S = np.zeros((len(K32), 16, 16))
    curve = [mse(S)]
    for _ in range(PASSES):
        for i in range(N):
            k = K32[:, i]
            S = S + beta * k[:, :, None] * (V32[:, i] - np.einsum("bi,bij->bj", k, S))[:, None, :]
        curve.append(mse(S))
    curves[beta] = curve
    print(
        f"β = {vn(beta, 1)}: sai số 1 lượt {vn(curve[1], 3)}, {PASSES} lượt {vn(curve[-1], 3)}; đọc đúng {pct(acc32(S), 1)}"
    )
S_add = write("add", K32, V32)
print(f"bình phương tối thiểu: sai số {vn(mse(S_ls), 3)}, đúng {pct(acc32(S_ls), 1)}")
print(f"S = 0: sai số {vn(mse(0 * S_ls), 3)} · phép cộng: sai số {vn(mse(S_add), 3)}, đúng {pct(acc32(S_add), 1)}")
assert curves[0.1][-1] < curves[0.3][-1] < curves[1.0][-1] and curves[0.1][-1] - mse(S_ls) < 0.01

Hình dưới vẽ sai số bình phương trung bình trên 32 cặp sau mỗi lượt quét, cho ba giá trị $\beta$ (màu và kiểu điểm
khác nhau); đường chấm đen là nghiệm bình phương tối thiểu, đáy mà không cách ghi tuyến tính nào xuống thấp hơn.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.4))
for (beta, curve), (color, marker) in zip(curves.items(), ((BLUE, "o"), (ORANGE, "^"), (GREEN, "D"))):
    ax.plot(
        np.arange(PASSES + 1),
        curve,
        color=color,
        marker=marker,
        ms=4,
        markevery=3,
        lw=1.8,
        label=f"delta, β = {vn(beta, 1)}",
    )
ax.axhline(mse(S_ls), color="black", ls=":", lw=1.5, label="bình phương tối thiểu")
ax.set(xlabel="số lượt quét qua 32 cặp", ylabel="½‖kS − v‖² trung bình", xlim=(0, PASSES), ylim=(0, 0.6))
ax.set_title("Quy tắc delta lặp lại nhiều lượt = SGD trên bài hồi quy khóa → giá trị")
ax.legend(frameon=False, ncols=2)
ax.grid(alpha=0.3)
vn_axes(fig)
plt.show()

Với $\beta = 1$, mỗi bước xóa sạch sai số của cặp đang ghi, nên nó đuổi theo cặp mới nhất và không bao giờ hội tụ (sai số
dao động quanh mức của $S = 0$). $\beta$ nhỏ hơn thì chậm hơn nhưng tiến về đáy: tốc độ học quá lớn làm SGD không hội tụ,
đúng như ở Bài 05. Một lớp DeltaNet chỉ đi **một lượt** qua dãy, nên nó không bao giờ đạt nghiệm bình phương tối thiểu;
nó học $\beta_t$ để quyết định mỗi token đáng sửa bao nhiêu.

### Khi một khóa được ghi lại

Chỗ quy tắc delta thật sự khác biệt là khi **cùng một khóa** được ghi lần thứ hai với giá trị mới. Ghi 16 cặp, rồi ghi lại
8 khóa trong số đó với giá trị mới; câu trả lời đúng là giá trị *mới nhất*. Phép cộng giữ cả hai giá trị chồng lên nhau,
còn quy tắc delta trừ giá trị cũ trước khi ghi.

In [ ]:
rng = np.random.default_rng(0)
D, N, m = len(keys), 16, 8
pick = np.argsort(rng.random((D, N)), 1)[:, :m]  # 8 khóa sẽ được ghi lại
new_ids = rng.integers(0, CODEWORDS, (D, m))
K_new = np.take_along_axis(keys[:, :N], pick[..., None], 1)
V_new = np.take_along_axis(code, new_ids[..., None], 1)
target = ids[:, :N].copy()
np.put_along_axis(target, pick, new_ids, 1)
is_new = np.zeros((D, N), bool)
np.put_along_axis(is_new, pick, True, 1)
for rule in ("add", "delta"):
    S = write(rule, np.concatenate([keys[:, :N], K_new], 1), np.concatenate([vals[:, :N], V_new], 1))
    got = (keys[:, :N] @ S @ code.transpose(0, 2, 1)).argmax(-1)
    ok = got == target
    old = (got == ids[:, :N])[is_new].mean()
    print(
        f"{RULES[rule]:<14}: đúng {pct(ok.mean(), 1)} · khóa ghi lại {pct(ok[is_new].mean(), 1)} "
        f"(vẫn đọc ra giá trị cũ: {pct(old, 1)}) · khóa giữ nguyên {pct(ok[~is_new].mean(), 1)}"
    )

Phép cộng không biết giá trị nào mới hơn: với khóa được ghi lại, nó đọc ra giá trị mới (36,0%) cũng thường như giá trị cũ
(36,7%). Quy tắc delta xóa giá trị cũ trước khi ghi, nên chỉ 2,0% khóa ghi lại còn đọc ra giá trị cũ, và 86,4% đọc ra giá
trị mới. Cái giá là các khóa *giữ nguyên* tụt từ 67,1% xuống 41,8%: mỗi lần sửa theo hướng $k_t$ cũng làm lệch các khóa
cũ chồng lên hướng đó. Hai hiệu ứng này chính là điều mà huấn luyện phải cân bằng, bằng cách học $\beta_t$ và học các
khóa ít chồng lấn hơn.

## 7 · Sau huấn luyện: trí nhớ vẫn có trần

Bộ nhớ chưa huấn luyện dùng khóa ngẫu nhiên. Một lớp được huấn luyện thì tự học khóa, giá trị, truy vấn và các cổng. Liệu
nó có vượt được trần đó không? Tệp `linear-recall.json` chứa kết quả của đúng bài toán ấy:
- mỗi dãy là $N$ cặp (khóa, giá trị) với khóa khác nhau, chọn trong 64 khóa và 64 giá trị (mỗi token mã hóa one-hot);
- một lớp trộn token duy nhất đọc cả dãy, rồi mỗi khóa được hỏi lại một lần;
- mỗi cấu hình học 6.000 bước (Adam, lô 64 dãy mới mỗi bước) với 3 seed. Tốc độ học $\eta$ được seed 0 chọn trên 1.000
  dãy kiểm định riêng, và độ chính xác đo trên 1.000 dãy kiểm tra khác.

Bảy lớp trộn token được so sánh: softmax attention (giữ mọi cặp trong bộ nhớ đệm), linear attention gốc (elu + 1, có mẫu
số), cộng dồn không cổng, có cổng quên, Mamba-2 (SSD), quy tắc delta và delta có cổng (Gated DeltaNet). Sáu lớp sau đọc qua
một trạng thái $16 \times 16$.

### 🤔 Dự đoán trước
Sau huấn luyện, với **64 cặp**, lớp tốt nhất có trạng thái $16 \times 16$ đúng khoảng bao nhiêu phần trăm? Còn softmax
attention thì sao?

In [ ]:
ARCHS = {
    "softmax": "softmax (bộ đệm KV)",
    "gated": "có cổng quên",
    "gdelta": "delta có cổng",
    "delta": "quy tắc delta",
    "ssd": "Mamba-2 (SSD)",
    "add": "cộng dồn",
    "linear": "linear gốc (elu + 1)",
}
N_TRAINED = (4, 8, 16, 32, 64)


def accs(task, arch, N, dk=16, fixed=False):
    """Độ chính xác kiểm tra của 3 seed, đọc từ tệp kết quả của demo."""
    run = REC["runs"][f"{task}-{arch}{'-fixed' if fixed else ''}-N{N}-d{dk}"]
    return np.array([s["acc"] for s in run["seeds"]])


def span(v, digits=1):
    """Khoảng min–max giữa các seed, dạng 79,0–79,9%."""
    lo, hi = vn(100 * v.min(), digits), vn(100 * v.max(), digits)
    return f"{lo}%" if lo == hi else f"{lo}–{hi}%"


print("số cặp N:            " + "".join(f"{N:>14}" for N in N_TRAINED))
for arch, label in ARCHS.items():
    print(f"{label:<21}" + "".join(f"{span(accs('recall', arch, N)):>14}" for N in N_TRAINED))
best3 = ("gated", "delta", "gdelta")
hold = [N for N in N_TRAINED if min(accs("recall", a, N).min() for a in best3) >= 0.99]
pool = lambda N: np.concatenate([accs("recall", a, N) for a in best3])
print(
    f"\nba lớp có cổng / delta: ≥ 99% đến N = {max(hold)}; 32 cặp {span(pool(32))} (≈ {span(pool(32), 0)}); "
    f"64 cặp {span(pool(64))} (≈ {span(pool(64), 0)})"
)
print(f"linear gốc: 32 cặp {span(accs('recall', 'linear', 32), 0)}, 64 cặp {span(accs('recall', 'linear', 64), 0)}")
print(
    f"bộ nhớ chưa huấn luyện, bình phương tối thiểu: 32 cặp {pct(table['ls'][31], 1)}, 64 cặp {pct(table['ls'][63], 1)}"
)
soft = np.concatenate([accs("recall", "softmax", N) for N in N_TRAINED])
print(
    f"softmax: {span(soft, 0)} ở mọi N; ở 64 cặp bộ đệm 64 × 32 = {vn(64 * 32, 0)} số, gấp {64 * 32 // 256} lần trạng thái"
)

Hình dưới vẽ cùng bảng đó: độ chính xác kiểm tra theo số cặp $N$ (trục ngang thang log 2), trung bình 3 seed, thanh dọc
là khoảng min–max. Mỗi lớp có màu và kiểu điểm riêng; hai đường chấm mảnh là bộ nhớ **chưa huấn luyện** của mục 5 (cộng
và bình phương tối thiểu) để so sánh.

In [ ]:
TRAINED_STYLE = {
    "softmax": (BLUE, "o", "-"),
    "gated": (ORANGE, "^", "-"),
    "gdelta": (GREEN, "D", "-"),
    "linear": (GREY, "s", "--"),
}
fig, ax = plt.subplots(figsize=(8, 4))
x = np.array(N_TRAINED)
for arch, (color, marker, ls) in TRAINED_STYLE.items():
    A = 100 * np.array([accs("recall", arch, N) for N in N_TRAINED])
    m = A.mean(1)
    ax.errorbar(
        x, m, yerr=[m - A.min(1), A.max(1) - m], color=color, marker=marker, ls=ls, lw=1.8, capsize=3, label=ARCHS[arch]
    )
for rule, color in (("ls", "black"), ("add", GREY)):
    ax.plot(x, 100 * table[rule][x - 1], color=color, ls=":", lw=1.2, label=f"chưa huấn luyện: {RULES[rule]}")
ax.set_xscale("log", base=2)
ax.set(xlabel="số cặp N", ylabel="đúng trên tập kiểm tra (%)", ylim=(0, 103))
ax.set_title("Tra cứu khóa–giá trị sau 6.000 bước, trạng thái 16 × 16 (3 seed)")
ax.legend(frameon=False, fontsize=8.5, loc="lower left")
ax.grid(alpha=0.3)
vn_axes(fig)
ax.set_xticks(x, [str(n) for n in N_TRAINED])  # sau vn_axes: vạch đúng ở các N đã huấn luyện
ax.xaxis.set_minor_locator(NullLocator())
plt.show()

Sau huấn luyện, các lớp có cổng hoặc quy tắc delta đúng ≥ 99% đến 8 cặp, rồi rơi xuống 79–80% ở 32 cặp và 48–49% ở 64 cặp.
Cả năm lớp trạng thái cố định (cộng dồn, có cổng, Mamba-2, delta, delta có cổng) gần như trùng nhau: ở bài này, cách quản
lý trí nhớ không cứu được một trạng thái quá nhỏ. Linear attention gốc còn thấp hơn (66% và 37%). Softmax đúng 100% ở mọi
$N$, vì nó không nén gì: với 64 cặp, bộ đệm của nó lớn gấp 8 lần trạng thái.

Lỗi nằm ở đâu? Tệp kết quả lưu độ chính xác của từng câu hỏi theo thứ tự ghi (seed 0, 32 cặp). Ô dưới chia chúng thành 4
nhóm từ cũ đến mới, như ở mục 5, và in cổng $\alpha$ trung bình mà mô hình có cổng tự học trên bài này.

In [ ]:
for arch in ("gated", "add", "gdelta"):
    byq = np.array(REC["runs"][f"recall-{arch}-N32-d16"]["seeds"][0]["byq"])
    print(f"{ARCHS[arch]:<14} cũ → mới: " + " / ".join(pct(g.mean(), 1) for g in np.array_split(byq, 4)))
g32 = REC["inspect"]["recall-gated-N32"]
print(f"có cổng: α trung bình {vn(g32['alphaMean'], 5)}; cặp đầu tiên còn {pct(g32['alphaPow'], 1)} độ lớn ở cuối dãy")

Ở lớp có cổng, lỗi rải đều trên mọi nhóm (78–80% mỗi nhóm): mô hình tự học $\alpha \approx 0{,}99996$, tức gần như không quên,
vì quên ở đây chỉ làm mất thông tin. Lỗi là **chồng lấn**, không phải quên. Delta có cổng thì nghiêng về các cặp mới, giống bộ
nhớ chưa huấn luyện ở mục 5: mỗi lần ghi, nó sửa phần của khóa mới trùng với các khóa cũ.

Vì sao huấn luyện không vượt được trần? Một phần là hình học: mô hình tự chọn được khóa, nhưng phải đặt **64 khóa** (một cho
mỗi mã khóa) trong **16 chiều**, nên chúng không thể vuông góc nhau. Cận Welch (1974) cho biết với $n$ vectơ đơn vị trong $\mathbb{R}^d$, trung
bình của $\cos^2$ giữa hai vectơ khác nhau không nhỏ hơn $(n/d - 1)/(n - 1)$. Ô dưới so cận đó với khóa ngẫu nhiên, và đọc
từ tệp kết quả mức chồng lấn của các truy vấn mà mô hình học được.

In [ ]:
n_keys, d_k = 64, 16
welch = (n_keys / d_k - 1) / (n_keys - 1)
G = keys[:, :n_keys] @ keys[:, :n_keys].transpose(0, 2, 1)
off = G[:, ~np.eye(n_keys, dtype=bool)]
print(
    f"64 vectơ đơn vị trong 16 chiều: cos² trung bình ≥ {vn(welch, 4)} (cận Welch); khóa ngẫu nhiên: {vn((off**2).mean(), 4)}"
)
print(
    f"tức |cos| cỡ {vn(np.sqrt(welch), 3)} ở mức tốt nhất, so với {vn(np.sqrt((off**2).mean()), 3)} của khóa ngẫu nhiên"
)
qc, phi = REC["inspect"], REC["phi"]
print(
    f"truy vấn học được (32 cặp, seed 0): |cos| trung bình {vn(qc['qcos-gated']['abs'], 3)} (có cổng), "
    f"{vn(qc['qcos-add']['abs'], 3)} (cộng dồn); vectơ ngẫu nhiên: {vn(phi['absCosL2'], 3)}"
)
lq = qc["qcos-linear"]
print(
    f"truy vấn học được của linear gốc, sau φ: cos trung bình {vn(lq['mean'], 3)}, |cos| {vn(lq['abs'], 3)} (mọi cos đều dương)"
)
print(
    f"φ = elu + 1 luôn dương: hai khóa ngẫu nhiên có cos trung bình {vn(phi['cosElu'], 2)} (nhỏ nhất {vn(phi['cosEluMin'], 2)})"
)
assert abs(welch - 3 / 63) < 1e-12

Khóa ngẫu nhiên có $\cos^2$ trung bình $1/d_k = 0{,}0625$; khóa tốt nhất có thể cũng chỉ xuống được $0{,}0476$. Mô hình học
được truy vấn ít chồng lấn hơn một chút (|cos| 0,169 so với 0,203), nhưng không thể làm chúng vuông góc.

Linear attention gốc thì khác ở *dấu*. $\varphi = \operatorname{elu} + 1$ luôn dương, nên hai khóa ngẫu nhiên có cos trung bình
0,70. Huấn luyện kéo con số đó xuống (truy vấn học được chỉ còn cos 0,128), nhưng mọi tích $\varphi(q)\cdot\varphi(k_j)$ vẫn
dương. Khi đó đầu ra $o = \sum_j w_j v_j / \sum_j w_j$ là một trung bình có trọng số dương của *mọi* giá trị đã ghi: nhiễu từ
các cặp khác cộng dồn cùng dấu, không triệt tiêu nhau như với khóa có dấu. Đó là lý do các lớp hiện đại bỏ $\varphi$ và mẫu
số, để $q, k$ mang dấu.

Trí nhớ tăng theo $d_k$. Tệp kết quả có cùng bài 32 cặp với trạng thái $8 \times 8$ và $32 \times 32$:

In [ ]:
for dk in (8, 16, 32):
    row = " · ".join(
        f"{ARCHS[a]} {span(accs('recall', a, 32, dk), 2)}" for a in ("gated", "gdelta", "linear", "softmax")
    )
    print(f"{dk:>2} × {dk:<2} ({vn(dk * dk, 0):>5} số), 32 cặp: {row}")
print("từng seed, có cổng ở 8 × 8:", " · ".join(pct(x, 1) for x in accs("recall", "gated", 32, 8)))
small = np.concatenate([accs("recall", a, 32, 8) for a in ("gated", "gdelta", "linear")])
print(f"8 × 8, ba lớp, bỏ seed kẹt: {span(small[small > 0.1], 0)}; đoán mò: 1/64 = {pct(1 / 64, 1)}")

Gấp 4 số ô (32 × 32) thì 32 cặp lại đúng gần hết (99,98–99,99%); chia 4 (8 × 8) thì chỉ còn 18–23%, và một seed của
lớp có cổng còn kẹt ở 2,4%, gần mức đoán mò 1/64. Softmax thì đúng 100% ở cả ba cỡ, vì nó không phụ thuộc vào $d_k$ để
*chứa*, chỉ để *phân biệt*.

### Trần nằm ở ma trận, hay ở cách ghi?

Một chi tiết dễ bỏ qua: bình phương tối thiểu là ma trận tốt nhất theo **sai số bình phương**, không phải theo **tỉ lệ đọc
đúng**. Nếu ta được tự do chọn $S$ để mỗi $k_i S$ gần từ đúng *hơn mọi từ khác* (tối ưu cross-entropy của Bài 04 trên 64 từ,
bằng Adam, xuất phát từ nghiệm bình phương tối thiểu), một ma trận $16 \times 16$ chứa được bao nhiêu? Ô dưới thử trên 100 lần
rút đầu tiên.

In [ ]:
def fit_readout(K, V, c, y, steps=300, lr=0.01, scale=8.0):
    """Tìm S để argmax_w (k_i S)·c_w = y_i, bằng Adam trên cross-entropy; xuất phát từ nghiệm bình phương tối thiểu."""
    S = write("ls", K, V)
    m1, m2 = np.zeros_like(S), np.zeros_like(S)
    for t in range(1, steps + 1):
        z = scale * (K @ S @ c.transpose(0, 2, 1))
        P = np.exp(z - z.max(-1, keepdims=True))
        P /= P.sum(-1, keepdims=True)
        np.put_along_axis(P, y[..., None], np.take_along_axis(P, y[..., None], -1) - 1, -1)  # softmax − one-hot
        g = scale * K.transpose(0, 2, 1) @ P @ c / K.shape[1]
        m1, m2 = 0.9 * m1 + 0.1 * g, 0.999 * m2 + 0.001 * g**2
        S -= lr * (m1 / (1 - 0.9**t)) / (np.sqrt(m2 / (1 - 0.999**t)) + 1e-8)
    return read_back(S, c, K, y).mean()


D = 100
for N in (32, 64):
    acc_fit = fit_readout(keys[:D, :N], vals[:D, :N], code[:D], ids[:D, :N])
    acc_ls = read_back(write("ls", keys[:D, :N], vals[:D, :N]), code[:D], keys[:D, :N], ids[:D, :N]).mean()
    print(f"{N} cặp, 100 lần rút: bình phương tối thiểu {pct(acc_ls, 1)} → tối ưu cho tỉ lệ đọc đúng {pct(acc_fit, 1)}")

Cùng một ma trận $16 \times 16$, 32 cặp đúng **100%** khi $S$ được tối ưu riêng cho việc đọc lại, so với 78,3% của bình
phương tối thiểu và 79–80% của các lớp đã huấn luyện. Ở 64 cặp, 66,4% so với 48–49%. Vậy trần ở trên không hẳn là sức
chứa của ma trận. Nó là giới hạn của **cách ghi**: một lớp tuyến tính chỉ đi một lượt qua dãy, cộng các cập nhật hạng 1,
và không thể giải một bài tối ưu như ô trên trong lúc đọc. Đó là hướng của các lớp kiểu TTT (Sun và cộng sự, 2024): coi
trạng thái là một mô hình nhỏ được *học* ngay khi đọc. Quy tắc delta chính là trường hợp đơn giản nhất, một bước gradient
mỗi token.

## 8 · Tự huấn luyện một cổng quên

Các lớp trên chưa cần quên: mỗi dãy chỉ có một nhóm cặp. Bài **xóa đoạn cũ** (reset) thì cần. Mỗi dãy gồm 16 cặp, một dấu
phân cách, rồi 16 cặp *khác*, và chỉ hỏi các khóa của nhóm thứ hai. Nhóm đầu chỉ làm đầy trạng thái. Một mô hình biết xóa
trạng thái ở dấu phân cách sẽ chỉ còn 16 cặp phải nhớ; không biết xóa thì phải chịu chồng lấn của cả 32.

Ta tự viết một lớp có cổng quên, đúng cấu hình của tệp kết quả (`reel/linear/mixers.py` của khóa học). Mỗi token $x$ là
vectơ one-hot của mã khóa (64), one-hot của mã giá trị (64) và một cờ (1 ở dấu phân cách). Từ đó:

$$k = \frac{xW_k}{\lVert xW_k \rVert},\quad v = xW_v,\quad \alpha = \sigma(xw_\alpha + b_\alpha),\quad \beta = \sigma(xw_\beta + b_\beta),
\qquad S_T = \sum_{s} \Big(\prod_{r > s} \alpha_r\Big)\beta_s\, k_s^\top v_s$$

Mỗi câu hỏi là một truy vấn học được $q = W_q[\text{mã khóa}]$ (chuẩn hóa độ dài 1), đọc $o = qS_T$, rồi một lớp tuyến tính
$oW_o + b_o$ cho 64 điểm số và softmax cross-entropy (Bài 04). Với **cổng hằng số**, $\alpha$ và $\beta$ không phụ thuộc
token: $w_\alpha = w_\beta = 0$, chỉ học $b_\alpha, b_\beta$. Ô dưới sinh dữ liệu giống hệt `recall.py`.

In [ ]:
NK = NV = 64  # số mã khóa và mã giá trị
F = NK + NV + 1  # one-hot khóa, one-hot giá trị, cờ


def features(keys_id, vals_id, flag):
    """Mã −1 (dấu phân cách) không bật one-hot nào."""
    B_, T_ = keys_id.shape
    X = np.zeros((B_, T_, F))
    b_, t_ = np.nonzero(keys_id >= 0)
    X[b_, t_, keys_id[b_, t_]] = 1
    b_, t_ = np.nonzero(vals_id >= 0)
    X[b_, t_, NK + vals_id[b_, t_]] = 1
    X[..., -1] = flag
    return X


def make_task(rng, n, N, task):
    """n dãy → X (n, T, F), mã khóa được hỏi (n, N) và đáp án (n, N); giống recall.py của khóa học."""
    ks = np.argsort(rng.random((n, NK)), 1)[:, :N]  # N mã khóa khác nhau
    vs = rng.integers(0, NV, (n, N))
    if task == "recall":  # N cặp, hỏi lại tất cả
        return features(ks, vs, np.zeros((n, N))), ks.copy(), vs
    # reset: N cặp, dấu phân cách, N cặp với khóa mới; hỏi nhóm thứ hai (recall.py rút ks, vs ở trên trước, như ở đây)
    ks = np.argsort(rng.random((n, NK)), 1)[:, : 2 * N]
    v1, v2 = rng.integers(0, NV, (n, N)), rng.integers(0, NV, (n, N))
    sep = np.full((n, 1), -1)
    kk, vv = np.concatenate([ks[:, :N], sep, ks[:, N:]], 1), np.concatenate([v1, sep, v2], 1)
    flag = np.zeros(kk.shape)
    flag[:, N] = 1
    return features(kk, vv, flag), ks[:, N:].copy(), v2


X, Qk, Y = make_task(np.random.default_rng(2016), 1000, 16, "reset")  # tập kiểm tra của tệp kết quả
strip = REC["inspect"]["reset-gated"]["strip"]
key0 = [int(np.argmax(t[:NK])) if t[:NK].any() else -1 for t in X[0]]
print(f"X cỡ {X.shape}; dãy kiểm tra đầu tiên: mã khóa {key0[:6]} … dấu phân cách ở vị trí {key0.index(-1)}")
assert key0 == strip["key"], "bộ sinh dữ liệu phải giống recall.py"

Mô hình và lan truyền ngược viết tay (Bài 06). Vì chỉ đọc trạng thái cuối, $S_T$ là một tổng có trọng số
$e_s = w_s \beta_s$ với $w_s = \prod_{r>s}\alpha_r = e^{c_T - c_s}$, nên không cần vòng lặp theo thời gian: gradient theo
$\log\alpha$ đi qua tổng tích lũy $c$. Ba kiểu cổng: `"token"` (học $w_\alpha, w_\beta$), `"fixed"` (chỉ $b_\alpha, b_\beta$),
`"none"` ($\alpha = \beta = 1$, cộng dồn).

In [ ]:
def sigmoid(z):
    return 0.5 * (1 + np.tanh(0.5 * z))  # cùng giá trị, không tràn số khi |z| lớn


def init(dk, seed):
    """Như mixers.init của khóa học: b_α = 3 nên lúc đầu α ≈ 0,95; b_β = 0 nên β = 0,5."""
    r = np.random.default_rng(seed)
    Wk, Wv = r.standard_normal((F, dk)) / np.sqrt(3), r.standard_normal((F, dk)) / np.sqrt(3)
    Wq, Wo = r.standard_normal((NK, dk)), r.standard_normal((dk, NV)) / np.sqrt(dk)
    zero = np.zeros(F)
    return {"Wk": Wk, "Wv": Wv, "wa": zero, "ba": np.array([3.0]), "wb": zero.copy(), "bb": np.array([0.0]),
            "Wq": Wq, "Wo": Wo, "bo": np.zeros(NV)}  # fmt: skip


def l2n(x):
    n = np.sqrt((x * x).sum(-1, keepdims=True) + 1e-12)
    return x / n, n


def forward(p, X, Qk, gate):
    B_, T_, _ = X.shape
    k, kn = l2n(X @ p["Wk"])
    q, qn = l2n(p["Wq"][Qk])
    v = X @ p["Wv"]
    if gate == "none":
        za, la, b = None, np.zeros((B_, T_)), np.ones((B_, T_))
    else:
        tok = gate == "token"
        za = X @ p["wa"] + p["ba"][0] if tok else np.full((B_, T_), p["ba"][0])
        zb = X @ p["wb"] + p["bb"][0] if tok else np.full((B_, T_), p["bb"][0])
        la, b = -np.logaddexp(0, -za), sigmoid(zb)  # log α = log σ(z_α), viết ổn định
    c = np.cumsum(la, 1)
    w = np.exp(c[:, -1:] - c)  # ∏_{r>s} α_r: phần của token s còn lại ở cuối dãy
    S = (k * (w * b)[..., None]).transpose(0, 2, 1) @ v  # S_T = Σ_s w_s β_s k_sᵀ v_s
    o = q @ S
    return {"k": k, "kn": kn, "q": q, "qn": qn, "v": v, "za": za, "la": la, "b": b, "w": w, "S": S, "o": o,
            "z": o @ p["Wo"] + p["bo"]}  # fmt: skip

In [ ]:
def l2n_back(g, y, n):
    """Gradient qua y = x/‖x‖: bỏ thành phần song song với y, rồi chia cho ‖x‖."""
    return (g - y * (g * y).sum(-1, keepdims=True)) / n


def loss_grad(p, X, Qk, Y, gate):
    f = forward(p, X, Qk, gate)
    z = f["z"] - f["z"].max(-1, keepdims=True)
    P = np.exp(z)
    P /= P.sum(-1, keepdims=True)
    loss = -np.log(np.take_along_axis(P, Y[..., None], -1)).mean()
    dz = P  # ∂L/∂z = (softmax − one-hot) / số câu hỏi
    np.put_along_axis(dz, Y[..., None], np.take_along_axis(dz, Y[..., None], -1) - 1, -1)
    dz /= Y.size
    k, q, v, w, b, S, o = (f[n] for n in ("k", "q", "v", "w", "b", "S", "o"))
    B_, T_, dk = k.shape
    g = {"Wo": o.reshape(-1, dk).T @ dz.reshape(-1, NV), "bo": dz.sum((0, 1))}
    do = dz @ p["Wo"].T
    G = q.transpose(0, 2, 1) @ do  # ∂L/∂S_T
    dq = l2n_back(do @ S.transpose(0, 2, 1), q, f["qn"])
    e = w * b
    kG = k @ G
    dk_ = e[..., None] * (v @ G.transpose(0, 2, 1))
    dv = e[..., None] * kG
    de = (kG * v).sum(-1)  # ∂L/∂e_s
    db, dlw = de * w, de * e  # e = w·β; ∂L/∂log w_s = ∂L/∂e_s · e_s
    dc = -dlw  # log w_s = c_T − c_s
    dc[:, -1] += dlw.sum(1)
    dla = np.cumsum(dc[:, ::-1], 1)[:, ::-1]  # c_t = Σ_{u≤t} log α_u
    if gate == "none":
        dza = dzb = np.zeros((B_, T_))
    else:
        dza, dzb = dla * (1 - sigmoid(f["za"])), db * b * (1 - b)  # d log σ(z)/dz = 1 − σ(z); dσ/dz = σ(1 − σ)
    Xf = X.reshape(B_ * T_, F)
    g["Wk"] = Xf.T @ l2n_back(dk_, k, f["kn"]).reshape(-1, dk)
    g["Wv"] = Xf.T @ dv.reshape(-1, dk)
    tok = gate == "token"
    g["wa"], g["wb"] = (Xf.T @ dza.ravel(), Xf.T @ dzb.ravel()) if tok else (np.zeros(F), np.zeros(F))
    g["ba"], g["bb"] = np.array([dza.sum()]), np.array([dzb.sum()])
    g["Wq"] = np.zeros_like(p["Wq"])
    np.add.at(g["Wq"], Qk.ravel(), dq.reshape(-1, dk))
    return loss, g


# kiểm tra gradient bằng sai phân hữu hạn, trên một lô nhỏ với cổng theo token khác 0
rng = np.random.default_rng(3)
p = init(16, 1)
p["wa"], p["wb"] = rng.standard_normal(F) * 0.5, rng.standard_normal(F) * 0.5
Xs, Qs, Ys = make_task(rng, 4, 4, "reset")
_, g = loss_grad(p, Xs, Qs, Ys, "token")
worst = 0.0
for name in p:
    for idx in [np.unravel_index(j * 37 % p[name].size, p[name].shape) for j in range(4)]:
        old = p[name][idx]
        p[name][idx] = old + 1e-6
        lp = loss_grad(p, Xs, Qs, Ys, "token")[0]
        p[name][idx] = old - 1e-6
        lm = loss_grad(p, Xs, Qs, Ys, "token")[0]
        p[name][idx] = old
        num = (lp - lm) / 2e-6
        worst = max(worst, abs(num - g[name][idx]) / max(1e-8, abs(num) + abs(g[name][idx])))
print(f"sai số tương đối lớn nhất giữa gradient viết tay và sai phân hữu hạn: {sci(worst)}")
assert worst < 1e-5

Gradient viết tay khớp sai phân hữu hạn. Vòng huấn luyện giống `recall.py`: Adam ($0{,}9$ / $0{,}999$), mỗi bước một lô 64 dãy
mới, khởi động tuyến tính rồi giảm theo cosin, cắt chuẩn gradient ở 1. Tệp kết quả học 6.000 bước; ở đây ta chỉ học 800 bước
(100 bước khởi động thay vì 200) với $\eta$ mà tệp kết quả đã chọn trên tập kiểm định, đủ để thấy cổng học được gì (mỗi lần chạy vài giây).

In [ ]:
def train(task, N, gate, lr, steps=800, warm=100, seed=0, B=64):
    p = init(16, seed + 1)
    m1 = {n: np.zeros_like(x) for n, x in p.items()}
    m2 = {n: np.zeros_like(x) for n, x in p.items()}
    rng = np.random.default_rng(seed + 100)  # dữ liệu huấn luyện: luôn là dãy mới
    for s in range(1, steps + 1):
        X, Qk, Y = make_task(rng, B, N, task)
        _, g = loss_grad(p, X, Qk, Y, gate)
        gn = np.sqrt(sum((x**2).sum() for x in g.values()))
        if gn > 1:  # cắt chuẩn gradient
            g = {n: x / gn for n, x in g.items()}
        lr_s = lr * min(1, s / warm) * 0.5 * (1 + np.cos(np.pi * s / steps))
        for n in p:
            m1[n] = 0.9 * m1[n] + 0.1 * g[n]
            m2[n] = 0.999 * m2[n] + 0.001 * g[n] ** 2
            p[n] -= lr_s * (m1[n] / (1 - 0.9**s)) / (np.sqrt(m2[n] / (1 - 0.999**s)) + 1e-8)
    return p


def test_acc(p, gate, task="reset", N=16):
    X, Qk, Y = make_task(np.random.default_rng(2000 + N), 1000, N, task)  # đúng 1.000 dãy kiểm tra của tệp kết quả
    f = forward(p, X, Qk, gate)
    return (f["z"].argmax(-1) == Y).mean(), f, X


LR = {g: REC["runs"][f"reset-gated{'' if g == 'token' else '-fixed'}-N16-d16"]["lr"] for g in ("token", "fixed")}
models, live = {}, {}
for gate in ("token", "fixed"):
    models[gate] = train("reset", 16, gate, LR[gate])
    acc, f, Xt = test_acc(models[gate], gate)
    alpha = np.exp(f["la"])
    live[gate] = (acc, alpha, f["b"])
    print(
        f"cổng {'theo token' if gate == 'token' else 'hằng số':<10} (η = {vn(LR[gate], 2)}): đúng {pct(acc, 1)} · "
        f"α ở dấu phân cách {vn(alpha[:, 16].mean(), 3)}, ở token khác {vn(np.delete(alpha, 16, 1).mean(), 4)}"
    )
ins = REC["inspect"]["reset-gated"]
print(
    f"tệp kết quả (6.000 bước, seed 0): α ở dấu phân cách {vn(ins['alphaSep'], 4)}, ở token khác "
    f"{vn(ins['alphaOther'], 4)}; α của cổng hằng số {vn(ins['fixedAlpha'], 3)}"
)
print(f"3 seed: theo token {span(accs('reset', 'gated', 16))} · hằng số {span(accs('reset', 'gated', 16, fixed=True))}")
print(f"không cổng (cộng dồn): {span(accs('reset', 'add', 16))} · softmax: {span(accs('reset', 'softmax', 16))}")
acc_tok, a_tok, _ = live["token"]
a_fix = live["fixed"][1][0, 0]
print(
    f"cổng hằng số α = {vn(a_fix, 3)}: ở cuối dãy, nhóm đầu còn {vn(a_fix**33, 2)}–{vn(a_fix**17, 2)} độ lớn, nhóm sau {vn(a_fix**15, 2)}–1"
)
assert acc_tok > live["fixed"][0] + 0.05 and a_tok[:, 16].mean() < 0.05 and np.delete(a_tok, 16, 1).mean() > 0.99

Sau 800 bước, mô hình có cổng theo token đã tự học cách **xóa**: $\alpha$ ở dấu phân cách đã xuống 0,016, còn ở mọi token khác
$\alpha \approx 0{,}998$, tức gần như không quên. Không ai dạy nó dấu phân cách nghĩa là gì; mất mát chỉ đo các câu trả lời
về nhóm thứ hai. Nó đúng 96,6%, gần bằng mức của bài 16 cặp không có nhóm đầu (96,9–97,1% ở mục 7), tức là nhóm đầu gần như
không còn để lại nhiễu. Cổng hằng số chỉ chọn được một $\alpha \approx 0{,}966$ cho mọi token, một sự thỏa hiệp: nhóm đầu
chỉ co lại còn 0,32–0,55 độ lớn, trong khi cặp cũ nhất của nhóm sau cũng đã mờ còn 0,59. Tệp kết quả, học 6.000 bước, cho
cùng bức tranh với $\alpha$ ở dấu phân cách còn nhỏ hơn (0,0036).

Hình dưới vẽ $\alpha_t$ và $\beta_t$ theo vị trí token trên dãy kiểm tra đầu tiên: chấm tròn xanh là mô hình vừa học ở trên,
vuông cam là mô hình seed 0 của tệp kết quả. Vạch xám là dấu phân cách.

In [ ]:
_, f_tok, _ = test_acc(models["token"], "token")
a_live, b_live = np.exp(f_tok["la"][0]), f_tok["b"][0]
pos = np.arange(len(a_live))
fig, axes = plt.subplots(2, 1, figsize=(9, 4.4), sharex=True)
for ax, live_v, saved, name in (
    (axes[0], a_live, strip["alpha"], "αₜ (cổng quên)"),
    (axes[1], b_live, strip["beta"], "βₜ (độ mạnh ghi)"),
):
    ax.axvline(16, color=GREY, lw=6, alpha=0.3)
    ax.plot(pos, live_v, "o-", color=BLUE, ms=5, lw=1.2, label="vừa học (800 bước)")
    ax.plot(pos, saved, "s", color=ORANGE, ms=4, mfc="none", mew=1.4, label="tệp kết quả (6.000 bước)")
    ax.set(ylabel=name, ylim=(-0.05, 1.08))
    ax.grid(alpha=0.3)
axes[0].legend(frameon=False, loc="lower left", ncols=2)
axes[0].set_title("Cổng mô hình tự học trên bài xóa đoạn cũ: 16 cặp | dấu phân cách | 16 cặp")
axes[1].set_xlabel("vị trí token (16 = dấu phân cách)")
plt.tight_layout()
vn_axes(fig)
plt.show()
print(f"β ở dấu phân cách: vừa học {vn(b_live[16], 3)}, tệp kết quả {vn(ins['betaSep'], 3)}")
print(f"β ở cặp thật: vừa học ≈ {vn(np.delete(b_live, 16).mean(), 2)}, tệp kết quả ≈ {vn(ins['betaOther'], 2)}")

$\beta_t$ cũng tự tắt ở dấu phân cách (không có gì để ghi). Hai mô hình học $\beta$ ở các cặp thật khác nhau (0,93 so với
0,74), vì độ lớn chung của $S$ có thể dồn vào $W_v$ hoặc $W_o$: chỉ dạng của đường cong là có nghĩa.

### Ghi đè một khóa, và lọc nhiễu

Hai bài khó hơn trong tệp kết quả (16 cặp, cùng trạng thái $16 \times 16$, 3 seed):
- **ghi lại:** 16 cặp, rồi 8 khóa trong số đó được ghi lại với giá trị mới; câu trả lời đúng là giá trị mới nhất (như thí
  nghiệm chưa huấn luyện ở mục 6);
- **nhiễu:** 16 cặp thật xen với 16 token nhiễu có cờ, khóa và giá trị ngẫu nhiên; phải bỏ qua nhiễu.

Softmax “có vị trí” (`softpos`) là softmax cộng thêm một độ dốc theo vị trí học được (kiểu ALiBi), nên nó biết token nào
đến sau.

In [ ]:
OVERWRITE = {"add": "cộng dồn", "gated": "có cổng", "delta": "quy tắc delta", "gdelta": "delta có cổng",
             "linear": "linear gốc", "softmax": "softmax", "softpos": "softmax có vị trí"}  # fmt: skip
print(f"{'ghi lại 8 trên 16 khóa':<22}{'tất cả':>14}{'khóa ghi lại':>14}{'khóa giữ nguyên':>17}")
for arch, label in OVERWRITE.items():
    seeds = REC["runs"][f"overwrite-{arch}-N16-d16"]["seeds"]
    new, kept = (np.array([s[k] for s in seeds]) for k in ("accNew", "accKept"))
    print(f"{label:<22}{span(accs('overwrite', arch, 16)):>14}{span(new):>14}{span(kept):>17}")
soft_new = np.array([s["accNew"] for s in REC["runs"]["overwrite-softmax-N16-d16"]["seeds"]])
print(
    f"làm tròn: delta {span(accs('overwrite', 'delta', 16), 0)}, có cổng {span(accs('overwrite', 'gated', 16), 0)}; "
    f"softmax: khóa ghi lại ≈ {span(soft_new, 0)}, tất cả ≈ {span(accs('overwrite', 'softmax', 16), 0)}"
)
blind = 0.5 * 1.0 + 0.5 * 0.5  # nửa số khóa giữ nguyên (đúng hết), nửa ghi lại (đoán đúng giá trị mới một nửa số lần)
print(f"mức của một mô hình không phân biệt giá trị cũ với mới: {pct(blind, 0)}")
noise, fixed = REC["inspect"]["noise-gated"], REC["inspect"]["noise-gated-fixed"]
print(
    f"\nnhiễu: có cổng theo token {span(accs('noise', 'gated', 16))}, cổng hằng số "
    f"{span(accs('noise', 'gated', 16, fixed=True))}, softmax {span(accs('noise', 'softmax', 16))}"
)
print(f"  cổng theo token (seed 0): β ở nhiễu {vn(noise['betaNoise'], 3)}, ở cặp thật {vn(noise['betaReal'], 3)}")
print(
    f"  cổng hằng số (seed 0): β = {vn(fixed['betaReal'], 3)} cho mọi token; |cos| giữa truy vấn và khóa nhiễu "
    f"{vn(fixed['cosNoise'], 3)}, khóa thật {vn(fixed['cosReal'], 3)}"
)

Ở bài ghi lại, quy tắc delta (và delta có cổng) đúng 91%, còn cổng quên chỉ 78%. Cổng co *cả* trạng thái, nên không thể xóa
riêng giá trị cũ của một khóa mà giữ các khóa khác; quy tắc delta thì sửa đúng hướng $k_t$. Softmax không có vị trí giữ
đúng 100% khóa giữ nguyên nhưng chỉ 50–51% khóa ghi lại: nó thấy cả hai giá trị và không biết giá trị nào mới hơn, nên
dừng ở $\tfrac12 \cdot 100\% + \tfrac12 \cdot 50\% = 75\%$. Thêm vị trí, softmax lại đúng 100%. Cổng quên ở đây cũng chỉ hơn
mức 75% đó một chút.

Ở bài nhiễu, cổng theo token học $\beta \approx 0$ cho token nhiễu: không ghi chúng vào trạng thái. Cổng hằng số không làm
vậy được, nên nó tìm cách khác: đặt khóa nhiễu gần như vuông góc với mọi truy vấn (|cos| 0,025 so với 0,208), tức dành một
phần không gian 16 chiều làm “thùng rác”. Đó là điều Mamba thêm vào S4: làm cổng phụ thuộc vào token (“chọn lọc”).

## 9 · Mamba và Mamba-2: cùng một phép tính, nhìn từ phía mô hình không gian trạng thái

Mamba (Gu & Dao, 2023) không xuất phát từ attention mà từ **mô hình không gian trạng thái** (SSM) rời rạc. Theo quy ước hàng,
với một đầu có $P$ kênh ($x_t$ là $1 \times P$) và trạng thái $h_t$ cỡ $N \times P$:

$$h_t = \bar A_t\, h_{t-1} + \Delta_t\, B_t^\top x_t, \qquad y_t = C_t\, h_t, \qquad \bar A_t = e^{-\Delta_t a}$$

- S4, trước Mamba, dùng $\Delta, B, C$ cố định: mô hình bất biến theo thời gian, nên cả lớp là một **phép tích chập** nhân quả
  với một nhân cố định, và huấn luyện được bằng tích chập.
- Mamba cho $B_t, C_t$ và bước $\Delta_t$ phụ thuộc vào token (“chọn lọc”). $\bar A_t$ trở thành một cổng quên phụ thuộc token,
  nhưng phép tích chập không còn dùng được nữa; Mamba bù lại bằng một **phép quét song song** viết riêng cho GPU. Ở Mamba,
  $A$ là ma trận đường chéo: mỗi chiều trạng thái quên với tốc độ riêng.
- Mamba-2 (Dao & Gu, 2024) thu $A$ về **một số** $a$ cho mỗi đầu. Khi đó đổi tên $(C, B, x) \to (q, k, v)$, $\alpha_t = e^{-\Delta_t a}$,
  $\beta_t = \Delta_t$, và ta được đúng linear attention có cổng của mục 4. Các tác giả gọi đó là *đối ngẫu không gian trạng
  thái* (SSD): dạng hồi tiếp và dạng ma trận $(L \odot CB^\top)x$ là cùng một mô hình.

Ô dưới chạy một lớp SSD theo đúng công thức SSM, rồi chạy hàm `parallel` của mục 4 với các tên đã đổi. Sau đó nó cho $\Delta, B, C$
cố định và so lớp đó với một phép tích chập.

In [ ]:
def ssm_scan(x, Bm, Cm, dt, a):
    """SSM chọn lọc với A = −a·I (Mamba-2): h_t = exp(−Δ_t a) h_{t−1} + Δ_t B_tᵀ x_t, y_t = C_t h_t."""
    h, y = np.zeros((Bm.shape[1], x.shape[1])), np.zeros_like(x)
    for t in range(len(x)):
        h = np.exp(-dt[t] * a) * h + dt[t] * np.outer(Bm[t], x[t])
        y[t] = Cm[t] @ h
    return y


rng = np.random.default_rng(0)
T, Nst, P, a_ssm = 64, 16, 8, 0.5
x = rng.standard_normal((T, P))
Bm, Cm = rng.standard_normal((T, Nst)), rng.standard_normal((T, Nst))
dt = np.logaddexp(0, rng.standard_normal(T) - 1)  # Δ_t = softplus(·) > 0, phụ thuộc token
y_ssm = ssm_scan(x, Bm, Cm, dt, a_ssm)
y_attn = parallel(Cm, Bm, x, np.exp(-dt * a_ssm), dt)  # (C, B, x) → (q, k, v), α = exp(−Δa), β = Δ
print(
    f"SSM chọn lọc (Mamba-2) và linear attention có cổng, dạng ma trận: lệch nhiều nhất {sci(np.abs(y_ssm - y_attn).max())}"
)
assert np.abs(y_ssm - y_attn).max() < 1e-10

# Δ, B, C cố định (kiểu S4): y là tích chập nhân quả của x với nhân κ_j = Δ·(C·B)·exp(−Δa)^j
d0, b0, c0 = 0.4, Bm[0], Cm[0]
y_lti = ssm_scan(x, np.tile(b0, (T, 1)), np.tile(c0, (T, 1)), np.full(T, d0), a_ssm)
kernel = d0 * (c0 @ b0) * np.exp(-d0 * a_ssm) ** np.arange(T)
y_conv = np.stack([np.convolve(x[:, j], kernel)[:T] for j in range(P)], 1)
print(f"tham số cố định: SSM và tích chập với một nhân {T} hệ số lệch {sci(np.abs(y_lti - y_conv).max())}")
assert np.abs(y_lti - y_conv).max() < 1e-10

Hai cách viết trùng nhau đến sai số làm tròn. Mamba-2 *là* linear attention có cổng, chỉ được đặt tên theo truyền thống SSM;
nó có một ma trận attention ngầm $(L \odot CB^\top)$ dù không bao giờ tính ra. Khi các tham số không phụ thuộc token, cùng
lớp đó là một phép tích chập: đó là S4. “Chọn lọc” là điều làm Mamba khác S4, và mục 8 vừa cho thấy nó đáng giá ở đâu: khi
phải quên hoặc bỏ qua *tùy theo nội dung*.

## 10 · Mô hình lai: bộ nhớ khi đọc văn bản dài

Một trạng thái cố định không tra cứu chính xác được mọi thứ (mục 7), còn attention đầy đủ thì tốn bộ nhớ (mục 3). Nhiều mô
hình lớn gần đây giữ **vài lớp attention đầy đủ** để tra cứu, còn lại là lớp tuyến tính. Ô dưới ghi cấu hình của chúng, lấy từ
`config.json` đã công bố trên Hugging Face (cùng các bản mà demo dùng), rồi tính:
- bộ nhớ đệm KV mỗi token, chỉ của các lớp attention;
- trạng thái cố định của các lớp tuyến tính: ma trận $S$ của mọi đầu, cộng bộ đệm của phép tích chập ngắn (kernel 4, giữ 3
  token) mà các lớp này dùng trước khi tính $q, k, v$.

### 🤔 Dự đoán trước
Qwen3-Next-80B-A3B có 80 tỷ tham số, gấp 10 lần Llama 3 8B. Ở 262.144 token, bộ nhớ đệm của nó lớn hơn hay nhỏ hơn 32 GiB
của Llama 3 8B?

In [ ]:
HYBRID = {
    # Qwen/Qwen3-Next-80B-A3B-Instruct: 48 lớp, cứ lớp thứ 4 là attention (full_attention_interval 4); attention 2 đầu KV × 256;
    # Gated DeltaNet 32 đầu giá trị × (128 × 128), 16 đầu khóa × 128, tích chập kernel 4
    "Qwen3-Next-80B-A3B": {"attn": 12, "lin": 36, "kv": 2 * 2 * 256, "state": 32 * 128 * 128,
                           "conv": (2 * 16 * 128 + 32 * 128) * 3},
    # ai21labs/Jamba-v0.1: 32 lớp, attention ở lớp 4, 12, 20, 28 (period 8, offset 4); 8 đầu KV × 128 (4096 / 32 đầu);
    # Mamba: 2 × 4096 = 8192 kênh × d_state 16, tích chập kernel 4
    "Jamba v0.1": {"attn": 4, "lin": 28, "kv": 2 * 8 * 128, "state": 8192 * 16, "conv": 8192 * 3},
    # MiniMaxAI/MiniMax-Text-01: 80 lớp, cứ 8 lớp có 1 softmax attention (8 đầu KV × 128); lightning attention 64 đầu × 128²
    "MiniMax-Text-01": {"attn": 10, "lin": 70, "kv": 2 * 8 * 128, "state": 64 * 128 * 128, "conv": 0},
    # MiniMaxAI/MiniMax-M2: 62 lớp, đều là attention đầy đủ (8 đầu KV × 128)
    "MiniMax-M2": {"attn": 62, "lin": 0, "kv": 2 * 8 * 128, "state": 0, "conv": 0},
    # deepseek-ai/DeepSeek-V3.2: 61 lớp MLA, mỗi token mỗi lớp giữ 512 số nén + 64 số khóa RoPE; mỗi truy vấn chỉ đọc 2.048
    # token do bộ chỉ mục chọn (attention thưa), nhưng vẫn giữ mọi token
    "DeepSeek-V3.2": {"attn": 61, "lin": 0, "kv": 512 + 64, "state": 0, "conv": 0},
    "Llama 3 8B": {"attn": 32, "lin": 0, "kv": 2 * 8 * 128, "state": 0, "conv": 0},
    "Llama 2 7B": {"attn": 32, "lin": 0, "kv": 2 * 32 * 128, "state": 0, "conv": 0},
}  # fmt: skip


def memory(m, T, bytes_per_number=2):
    """Bộ đệm KV ở T token và trạng thái cố định, bằng byte (bf16)."""
    return m["attn"] * m["kv"] * T * bytes_per_number, m["lin"] * (m["state"] + m["conv"]) * bytes_per_number


ref = {m["name"]: m for m in COST["models"]}  # cùng các con số, do cost.py của khóa học tính từ config.json
for name, m in HYBRID.items():
    kv, st = memory(m, T_LONG)
    r = ref[name]["at"][str(T_LONG)]
    same = (kv, st) == (r["kv"], r["state"])
    print(
        f"{name:<19} {m['attn']:>2} attention + {m['lin']:>2} tuyến tính: {vn(kv / T_LONG / KiB, 1):>5} KiB mỗi token  →  "
        f"{vn(kv / GiB, 2):>6} GiB KV + {vn(st / MiB, 1):>5} MiB trạng thái = {vn((kv + st) / GiB, 2):>6} GiB  "
        f"(như tệp của demo: {same})"
    )
    assert same, "khác với demo: hãy kiểm tra lại cấu hình"
assert memory(HYBRID["Jamba v0.1"], T_LONG)[0] == 4 * GiB  # Bảng 1 của Jamba: 4GB ở 256K token, 16 bit
assert memory(HYBRID["Qwen3-Next-80B-A3B"], 1)[0] == 24 * KiB  # 12 lớp × 2 đầu KV × 256 × (K và V) × 2 byte

Bộ nhớ đệm của Qwen3-Next ở 262.144 token chỉ là 6 GiB, cộng khoảng 38 MiB trạng thái cố định: tổng 6,04 GiB, chưa bằng một
phần năm 32 GiB của Llama 3 8B, dù mô hình lớn gấp 10 lần. Hai lý do cộng lại: chỉ 12 trên 48 lớp giữ bộ nhớ đệm, và mỗi lớp
đó chỉ có 2 đầu KV. Jamba đúng 4 GiB như Bảng 1 trong bài báo của nó. Phần trạng thái cố định nhỏ đến mức gần như không thấy.

Bố cục lớp đọc từ tệp của demo (A: attention đầy đủ, S: attention thưa vẫn giữ mọi token, L: lớp tuyến tính):

In [ ]:
for name in (
    "Qwen3-Next-80B-A3B",
    "Jamba v0.1",
    "MiniMax-Text-01",
    "Kimi Linear 48B",
    "Nemotron-H-8B",
    "DeepSeek-V3.2",
):
    m = ref[name]
    n_a = sum(ch in "AS" for ch in m["layout"])
    print(f"{name:<19} {m['layout']}  ({n_a} attention / {len(m['layout'])} lớp; lớp tuyến tính: {m['mixer'] or '—'})")
assert ref["Qwen3-Next-80B-A3B"]["layout"] == "LLLA" * 12 and ref["Jamba v0.1"]["layout"].count("A") == 4

Hình dưới vẽ tổng bộ nhớ (KV cộng trạng thái, bf16) theo độ dài văn bản, cả hai trục thang log. Mỗi mô hình có màu và kiểu
điểm riêng. Trên thang log–log, bộ nhớ tỉ lệ với $T$ là một đường thẳng độ dốc 1; các mô hình chỉ khác nhau ở số byte mỗi token
(độ cao của đường). Một mô hình chỉ có lớp tuyến tính sẽ là một đường nằm ngang.

In [ ]:
Ts = np.array([2**k for k in range(10, 21)])
LINES = {
    "Llama 2 7B": (GREY, "s", "--"),
    "MiniMax-M2": (ORANGE, "v", "-"),
    "Llama 3 8B": (BLUE, "o", "-"),
    "DeepSeek-V3.2": ("black", "x", ":"),
    "Qwen3-Next-80B-A3B": (GREEN, "D", "-"),
    "Jamba v0.1": (GREEN, "^", "--"),
}
fig, ax = plt.subplots(figsize=(8.5, 4.2))
for name, (color, marker, ls) in LINES.items():
    total = [sum(memory(HYBRID[name], T)) / GiB for T in Ts]
    ax.plot(Ts, total, color=color, marker=marker, ls=ls, ms=5, lw=1.6, label=name)
ax.axvline(T_LONG, color=GREY, lw=1, ls=":")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set(xlabel="độ dài văn bản T (token)", ylabel="bộ nhớ đệm + trạng thái (GiB)", xlim=(Ts[0], Ts[-1]))
ax.set_title("Bộ nhớ khi đọc văn bản dài, tính từ config.json (bf16)")
ax.legend(frameon=False, fontsize=9, loc="upper left")
ax.grid(alpha=0.3, which="major")
vn_axes(fig)
plt.show()
q, l3 = sum(memory(HYBRID["Qwen3-Next-80B-A3B"], T_LONG)), sum(memory(HYBRID["Llama 3 8B"], T_LONG))
print(
    f"ở {vn(T_LONG, 0)} token: Qwen3-Next {vn(q / GiB, 2)} GiB, Llama 3 8B {vn(l3 / GiB, 0)} GiB (gấp {vn(l3 / q, 1)} lần)"
)
kv1, st = memory(HYBRID["Qwen3-Next-80B-A3B"], 1)
print(f"Qwen3-Next: bộ đệm KV vượt trạng thái cố định ({vn(st / MiB, 1)} MiB) từ khoảng {vn(st / kv1, 0)} token")

Đường của Qwen3-Next cong ở đoạn ngắn: dưới khoảng 1.608 token, 37,7 MiB trạng thái cố định còn lớn hơn bộ đệm KV. Từ đó trở
đi bộ đệm của 12 lớp attention chiếm phần lớn, và đường lại thẳng. Ở mọi độ dài, nó nằm thấp hơn Llama 3 8B khoảng 5 lần.

Vì sao không bỏ hẳn attention? Mục 7 cho thấy một trạng thái cố định không tra cứu chính xác được khi có nhiều cặp. Jelassi
và cộng sự (2024) chứng minh điều tương tự cho việc chép chuỗi: Transformer hai lớp chép được chuỗi dài theo hàm mũ, còn mô
hình có trạng thái cố định bị giới hạn bởi kích thước trạng thái. Vài lớp attention đầy đủ lo phần tra cứu đó. Nhưng cuộc
đua chưa ngã ngũ:
- **Attention thưa:** DeepSeek-V3.2 vẫn giữ mọi token trong bộ nhớ đệm, nhưng mỗi truy vấn chỉ đọc 2.048 token do một bộ chỉ
  mục chọn. Nó bớt phép tính chứ không bớt bộ nhớ.
- **Quay lại attention đầy đủ:** MiniMax-M2 dùng attention đầy đủ ở cả 62 lớp, sau khi MiniMax-Text-01 dùng bản lai 7 : 1.
  Theo blog của MiniMax, bản lai ngang ngửa trên các benchmark cũ nhưng ở quy mô lớn kém rõ ở suy luận nhiều bước.
- **Tiếp tục lai:** Kimi Linear và Qwen3.5 giữ tỉ lệ khoảng 3 lớp tuyến tính cho 1 lớp attention.

In [ ]:
for name in ("Qwen3.5-397B-A17B", "Kimi Linear 48B"):
    m = ref[name]
    n_a, n_l = m["layout"].count("A"), m["layout"].count("L")
    print(f"{name:<19} {m['layout']}  ({n_l} {m['mixer']} : {n_a} attention ≈ {vn(n_l / n_a, 1)} : 1)")
print(
    f"DeepSeek-V3.2: mỗi truy vấn đọc {vn(ref['DeepSeek-V3.2']['topk'], 0)} token đã chọn, nhưng giữ cả {vn(T_LONG, 0)}"
)

## 11 · Thử thay đổi

Các ô dưới chạy lại cùng các hàm với tham số khác. Bạn có thể sửa giá trị rồi chạy lại.

### Cỡ trạng thái $d_k$ của bộ nhớ chưa huấn luyện
Mục 7 thấy trí nhớ đã huấn luyện tăng theo $d_k$. Bộ nhớ chưa huấn luyện có như vậy không? Ô dưới rút lại khóa và từ giá trị
trong $d_k$ chiều (với $d_v = d_k$; các lần rút khác demo, vì demo chỉ dùng $d_k = 16$).

In [ ]:
for dk in (8, 16, 32):
    c_, k_, i_, v_ = memory_draws(draws=200, dk=dk)
    row = []
    for N in (8, 16, 32, 64):
        a_add = read_back(write("add", k_[:, :N], v_[:, :N]), c_, k_[:, :N], i_[:, :N]).mean()
        a_ls = read_back(write("ls", k_[:, :N], v_[:, :N]), c_, k_[:, :N], i_[:, :N]).mean()
        row.append(f"N {N}: {pct(a_add, 0)} / {pct(a_ls, 0)}")
    print(f"d_k = {dk:>2} (cộng / bình phương tối thiểu): " + " · ".join(row))

Bình phương tối thiểu luôn khớp đúng đến $N = d_k$. Với phép cộng, ở $N = d_k$ tỉ lệ đúng là 55% ($d_k = 8$), 83% (16) và 96% (32):
trong nhiều chiều hơn, khóa ngẫu nhiên ít chồng lấn hơn ($\cos^2$ trung bình $1/d_k$), nên trí nhớ tăng nhanh hơn cả $d_k$.

### Cổng quên cố định: nhớ lâu hay nhớ ít?
Một cổng hằng số $\alpha$ đánh đổi giữa nhớ cặp cũ và giảm chồng lấn. Thử vài giá trị ở 32 cặp chưa huấn luyện, chia 4 nhóm
theo thứ tự ghi như mục 5:

In [ ]:
N = 32
for alpha in (0.8, 0.9, 0.95, 0.99, 1.0):
    h = read_back(write("gate", keys[:, :N], vals[:, :N], alpha=alpha), code, keys[:, :N], ids[:, :N])
    groups = " / ".join(pct(g.mean(), 0) for g in np.array_split(h, 4, axis=1))
    print(f"α = {vn(alpha, 2)}: đúng {pct(h.mean(), 1)}  (cũ → mới: {groups})")

Không giá trị $\alpha < 1$ nào vượt được phép cộng ($\alpha = 1$) trên bài này: quên chỉ có ích khi có thứ *đáng* quên, như
đoạn cũ ở mục 8. Một cổng hằng số không phân biệt được hai trường hợp; cổng theo token thì có.

### Huấn luyện lại với seed khác, hoặc không có cổng
Mỗi lần chạy vài giây. Đổi `seed` hoặc `gate` (`"token"`, `"fixed"`, `"none"`) rồi so với khoảng 3 seed của tệp kết quả.

In [ ]:
for gate, seed in (("token", 1), ("none", 0)):
    lr = REC["runs"]["reset-gated-N16-d16" if gate == "token" else "reset-add-N16-d16"]["lr"]
    p_ = train("reset", 16, gate, lr, seed=seed)
    acc, f_, _ = test_acc(p_, gate)
    ref_acc = accs("reset", "gated" if gate == "token" else "add", 16)
    print(
        f"cổng {gate:<5} seed {seed} (η = {vn(lr, 2)}): đúng {pct(acc, 1)}, α ở dấu phân cách "
        f"{vn(np.exp(f_['la'][:, 16]).mean(), 3)}  ·  tệp kết quả (6.000 bước, 3 seed): {span(ref_acc)}"
    )

Học 800 bước thì cả hai thấp hơn khoảng 3 seed của tệp kết quả một chút, nhưng thứ tự không đổi: cổng theo token vẫn tự tìm
ra dấu phân cách ở seed khác, còn cộng dồn không có cách nào xóa nhóm đầu.

## 12 · Giới hạn và bước tiếp theo

- **Bài toán đồ chơi, một lớp, một đầu.** Mô hình thật xếp chồng hàng chục lớp, mỗi lớp nhiều đầu, thêm tích chập ngắn trước
  $q, k, v$ và một cổng ở đầu ra. Tra cứu khóa–giá trị chỉ là một khía cạnh của mô hình hóa ngôn ngữ, dù là khía cạnh quan
  trọng: Arora và cộng sự ([2023](https://arxiv.org/abs/2312.04927)) thấy nó giải thích 82% khoảng cách perplexity giữa
  attention và các mô hình tích chập có cổng.
- **Không đo tốc độ.** Notebook chỉ đếm bộ nhớ. Lớp tuyến tính rẻ khi *sinh* (mỗi token cập nhật một trạng thái cố định), còn
  khi *huấn luyện* thì nhanh nhờ dạng theo khối chạy bằng nhân ma trận trên GPU, với các kernel viết riêng.
- **Trí nhớ cố định vẫn là trí nhớ cố định.** Không cổng hay quy tắc ghi nào làm một ma trận $16 \times 16$ giữ đúng 64 cặp
  tùy ý theo một lượt đọc. Mô hình lai giữ vài lớp attention đầy đủ chính vì thế.

Bài 22–25 nói về **mô hình nền**: tiền huấn luyện, quy luật mở rộng, MoE, và giờ là kiến trúc cho văn bản dài. Nhưng một mô
hình nền chỉ biết viết tiếp văn bản. Bài 26 biến nó thành một trợ lý: học có giám sát trên các câu trả lời mẫu, rồi học tăng
cường với phần thưởng mà một chương trình kiểm tra được.

## Tóm tắt
- Bộ nhớ đệm KV tăng tuyến tính theo độ dài: Llama 3 8B cần 128 KiB mỗi token, tức 32 GiB ở 262.144 token; Llama 2 7B, không
  có GQA, cần 128 GiB, đúng con số trong Bảng 1 của bài báo Jamba.
- Bỏ softmax, attention viết được thành RNN với trạng thái $S_t = \alpha_t S_{t-1} + \beta_t k_t^\top v_t$, $o_t = q_t S_t$. Dạng
  hồi tiếp, dạng ma trận có mặt nạ và dạng theo khối là cùng một phép tính, có hay không có quy tắc delta: lệch nhau cỡ
  $10^{-14}$ ở 2.048 token.
- Một ma trận $16 \times 16$ chưa huấn luyện ghi 16 cặp khóa ngẫu nhiên bằng phép cộng chỉ đọc lại đúng 83,1% vì khóa chồng
  lấn; khóa trực giao thì 100,0%. Bình phương tối thiểu khớp đúng đến 16 cặp, rồi chỉ còn 79,0% ở 32 cặp.
- Quy tắc delta là một bước gradient descent trên $\tfrac12\lVert k_t S - v_t\rVert^2$ với tốc độ học $\beta_t$; lặp lại với $\beta$
  nhỏ thì tiến về nghiệm bình phương tối thiểu.
- Sau huấn luyện, trạng thái $16 \times 16$: ≥ 99% đến 8 cặp, 79–80% ở 32 và 48–49% ở 64 (elu + 1: 66% và 37%; softmax: 100%).
  Trạng thái $32 \times 32$ đưa 32 cặp lên 99,98–99,99%.
- Cổng quên theo token tự học xóa đoạn cũ: $\alpha = 0{,}0036$ ở dấu phân cách (tệp kết quả), đúng 96,9–97,3% so với 88,6–88,9%
  của cổng hằng số. Khi một khóa được ghi lại, quy tắc delta đúng 91%, cổng quên chỉ 78%.
- Mamba-2 là linear attention có cổng với $(C, B, x) \to (q, k, v)$, $\alpha_t = e^{-\Delta_t a}$, $\beta_t = \Delta_t$; bỏ “chọn lọc”
  thì còn lại một phép tích chập (S4).
- Mô hình lai: Qwen3-Next xen 3 lớp Gated DeltaNet với 1 lớp attention, chỉ 24 KiB mỗi token. Ở 262.144 token nó cần 6,04 GiB,
  so với 32 GiB của Llama 3 8B.

## Câu hỏi ôn tập
1. Vì sao Llama 2 7B cần bộ nhớ đệm KV gấp 4 lần Llama 3 8B, dù ít tham số hơn?
2. Dạng hồi tiếp và dạng ma trận cho cùng một $O$. Khi sinh văn bản thì dùng dạng nào, khi huấn luyện thì dùng dạng nào, và vì
   sao RNN của Bài 15 không có dạng theo khối?
3. Ghi 16 cặp với khóa ngẫu nhiên vào một ma trận $16 \times 16$ bằng phép cộng chỉ đọc lại đúng 83,1%. Lỗi đến từ đâu, và khi
   nào thì đúng 100%?
4. Quy tắc delta là một bước gradient descent trên mất mát nào? Với $\beta = 1$ và $\lVert k \rVert = 1$, nó làm gì với cặp vừa ghi?
5. Ở bài ghi lại, vì sao cổng quên chỉ đúng 78% còn quy tắc delta đúng 91%? Vì sao softmax không có vị trí dừng ở khoảng 75%?
6. Mamba-2 khác linear attention có cổng ở đâu? Điều gì làm Mamba khác S4?

<details><summary>Gợi ý đáp án</summary>

1. Llama 3 dùng GQA: 8 đầu KV cho 32 đầu truy vấn, nên mỗi lớp chỉ giữ 8 cặp K, V mỗi token thay vì 32.
2. Sinh văn bản dùng dạng hồi tiếp: mỗi token mới chỉ cập nhật và đọc trạng thái cố định. Huấn luyện dùng dạng theo khối (hoặc
   ma trận): cả dãy đã có sẵn, nên tính song song bằng nhân ma trận. RNN của Bài 15 có $\tanh$ bên trong vòng lặp, nên $h_t$ không
   phụ thuộc tuyến tính vào $h_{t-1}$ và không thể gộp nhiều bước thành một phép nhân ma trận.
3. Đọc lại cho $o_i = v_i + \sum_{j \ne i} (k_i \cdot k_j) v_j$: khóa ngẫu nhiên không vuông góc nhau, nên các cặp khác cộng nhiễu vào.
   Với $N \le d_k$ khóa trực giao, mọi tích $k_i \cdot k_j$ ($i \ne j$) bằng 0 và đọc lại đúng tuyệt đối.
4. $\tfrac12\lVert k_t S - v_t\rVert^2$, với tốc độ học $\beta_t$. Với $\beta = 1$, $k_t S_t = v_t$: sai số trên cặp vừa ghi về đúng 0,
   giá trị cũ của $k_t$ bị xóa và thay bằng giá trị mới.
5. Cổng quên co *cả* trạng thái, nên không thể xóa riêng giá trị cũ của một khóa; quy tắc delta trừ đúng phần $k_t S_{t-1}$ của khóa
   đang ghi. Softmax không có vị trí thấy cả giá trị cũ lẫn mới và không biết cái nào sau: nó đúng mọi khóa giữ nguyên và khoảng
   một nửa khóa ghi lại.
6. Không khác gì: Mamba-2 là linear attention có cổng, chỉ đổi tên $(C, B, x) \to (q, k, v)$, với $\alpha_t = e^{-\Delta_t a}$ và
   $\beta_t = \Delta_t$. Mamba cho $\Delta, B, C$ phụ thuộc vào token (“chọn lọc”); S4 dùng tham số cố định, nên cả lớp là một phép
   tích chập.
</details>

## Tìm hiểu thêm
- A. Katharopoulos và cộng sự (2020), [Transformers are RNNs](https://arxiv.org/abs/2006.16236): linear attention với
  $\varphi = \operatorname{elu} + 1$.
- I. Schlag, K. Irie, J. Schmidhuber (2021), [Linear Transformers Are Secretly Fast Weight Programmers](https://arxiv.org/abs/2102.11174):
  giới hạn trí nhớ và quy tắc delta.
- S. Yang và cộng sự (2024), [Parallelizing Linear Transformers with the Delta Rule over Sequence Length](https://arxiv.org/abs/2406.06484),
  và blog [DeltaNet Explained](https://sustcsonglin.github.io/blog/2024/deltanet-1/): quy tắc delta như hồi quy trực tuyến, dạng UT.
- S. Yang, J. Kautz, A. Hatamizadeh (2024), [Gated Delta Networks](https://arxiv.org/abs/2412.06464).
- A. Gu, T. Dao (2023), [Mamba](https://arxiv.org/abs/2312.00752); T. Dao, A. Gu (2024), [Mamba-2 / SSD](https://arxiv.org/abs/2405.21060)
  và [blog phần I–IV](https://tridao.me/blog/2024/mamba2-part1-model/).
- S. Arora và cộng sự (2023), [Zoology](https://arxiv.org/abs/2312.04927) và (2024), [Based](https://arxiv.org/abs/2402.18668):
  tra cứu khóa–giá trị và đánh đổi giữa cỡ trạng thái và trí nhớ.
- S. Jelassi và cộng sự (2024), [Repeat After Me](https://arxiv.org/abs/2402.01032): Transformer chép chuỗi tốt hơn mô hình có
  trạng thái cố định.
- Y. Sun và cộng sự (2024), [Learning to (Learn at Test Time)](https://arxiv.org/abs/2407.04620): trạng thái là một mô hình học
  ngay khi đọc.
- L. Welch (1974), [Lower bounds on the maximum cross correlation of signals](https://doi.org/10.1109/TIT.1974.1055219).
- Mô hình lai và các hướng khác: [Jamba](https://arxiv.org/abs/2403.19887), [MiniMax-01](https://arxiv.org/abs/2501.08313),
  [Qwen3-Next](https://huggingface.co/Qwen/Qwen3-Next-80B-A3B-Instruct), [Kimi Linear](https://arxiv.org/abs/2510.26692),
  [DeepSeek-V3.2](https://huggingface.co/deepseek-ai/DeepSeek-V3.2) và
  [Why Did MiniMax M2 End Up as a Full Attention Model?](https://huggingface.co/blog/MiniMax-AI/why-did-m2-end-up-as-a-full-attention-model).

**Bài tiếp theo:** [Bài 26 · Từ mô hình ngôn ngữ đến trợ lý](https://d3qr1cjln6qqtk.cloudfront.net/bai-hoc/hau-huan-luyen/):
biến một mô hình chỉ biết viết tiếp văn bản thành một trợ lý trả lời đúng câu hỏi.

In [ ]:
import platform

import matplotlib

print(f"Python {platform.python_version()} · numpy {np.__version__} · matplotlib {matplotlib.__version__}")